# Swiss Citation — Anchor-Funnel v7.5 multi-query

Runs the full anchor-funnel pipeline on **every** query in `val.csv` with **no** per-id branching. `topk_final = 50,000` (effectively returns the post-gate channel union).

**Pass criterion: R@1000 ≥ 0.60** for val_001 (≥ 26/42 gold). Stretch: ≥ 0.90 / 38+.

This notebook is structured as **11 phases** with a markdown header before every code cell.
Each markdown header states:
- **What** the next cell computes
- **Why** we need that signal (which retrieval gap it closes)
- **Expected outputs** — numeric ranges so anomalies pop out
- **Failure modes** — what to check if a number looks off

The final phase (Phase 10) does **per-gold diagnosis**: for every gold citation
NOT captured in top-1000, the notebook traces through every channel and every
signal source and prints a concrete root-cause report (with rank, score, token
overlap, cosine similarity, graph degrees).

## Phase map

| Phase | Cells | Purpose |
|---|---|---|
| 1. Setup | env, drive, paths, knobs | runtime + IO + global config |
| 2. Load val + corpus indexes | val.csv, law-llm jsonl, court-v5 jsonl | build all in-memory indexes |
| 3. Citation graph | sqlite → idx_graph_out / idx_graph_in | 4-layer graph (23.65M edges) |
| 4. Per-area bedrock + co-citation | corpus statistics | universal articles per legal area |
| 5. BM25 (FTS5 in-memory) | query-side lexical match | catches code names + Swiss terminology |
| 6. Vector channel setup | Qwen3-Embedding-8B + corpus E_GPU | semantic similarity |
| 7. Query expansion (Qwen3-32B) | LLM → structured JSON targets | concepts_en, term_targets_de/fr, statute_targets, legal_area_keywords |
| 8. Run channels | 14 retrieval signals | each with per-channel R@K |
| 9. RRF fusion + gating | reciprocal rank fusion + neg-gate | top-1000 final pool |
| 10. **Diagnosis** | per-gold trace + miss attribution | which signal failed for which gold |
| 11. Save artifacts + cleanup | persist to Drive | reproducibility |

## Constraints respected (recall from prior chat)

- **No hardcoded lists.** No DE_LEX, no fixed BGE list, no statute cluster table.
- **No query-specific knowledge.** Architecture must generalize to any val/test/production query.
- **No train data.** Train.csv is never read.
- **Open-source only.** Qwen3-32B (query expansion), Qwen3-Embedding-8B (vectors).

## v7 changes vs v6 (R@1000 was 0.357)

| Change | Why |
|---|---|
| Sibling budget 500 → 2000 | v6's 500 cap was a non-deterministic `set→list[:budget]` slice; with 2691 typical seeds × ~5 siblings, the slice dropped val_001 sibling Es. |
| Graph forward channel (NEW) | Loads corpus-derived citation graph (4 alias passes, 23.65M edges) and follows outgoing edges — surfaces text-cited targets PLUS all sibling Es of cited judgments via case-level fan-out. Closes 6/11 originally-orphan val_001 gold. |
| Graph reverse channel (NEW) | Follows incoming edges — finds rows that text-cite seeds. Co-citation expansion via the actual corpus graph, not LLM. |
| Phase 10 diagnostics | Every missed gold gets a per-channel trace + recommended fix. |

---

## v7.5 channel summary (production-grade fixes)

| # | Channel | Index used | Gap closed | Budget | RRF weight |
|---|---|---|---|---:|---:|
| 1 | law_direct_match | idx_law_direct | LLM-named statutes (incl. corpus-related codes) → matching law rows. Uncapped. | None | 1.2 |
| 2 | court_statute | idx_court_statute | LLM-named statute appears in row's `statute_anchors`. Specificity-weighted + paragraph_role boost. | 8000 | 1.5 |
| 3 | co_citation | co_neighbours + idx_law_direct + idx_court_statute | Statute cluster co-cited with LLM target (e.g., 221 StPO ↔ 212 StPO). Specificity-weighted. | 2500 | 0.7 |
| 4 | per_area_bedrock | per_area_canon_count, filtered by corpus-derived code family | Universal procedural articles per legal area (Art. 100 BGG, Art. 422 StPO etc.). top_n=1000 captures deep procedural cluster. | 1500 | 1.5 |
| 5 | statute_backprop | doc_statute_anchors | Caught court rows → law articles they cite. Rare-canon specificity weight. | 2000 | 2.5 |
| 6 | sibling_expansion | idx_court_base + idx_judgment_importance | All Es of caught judgments. Judgment-importance scoring (BGE landmark > obscure docket). | 5000 | 1.0 |
| 7 | graph_forward (v7) | idx_graph_out + idx_judgment_importance | Caught row → text-cited targets + case-level fan-out. Importance-weighted. | 5000 | 2.0 |
| 8 | graph_reverse (v7) | idx_graph_in + idx_judgment_importance | Caught row ← rows that text-cite it. Landmark filter (judgment importance ≥ 5). | 3000 | 0.5 |
| 9 | graph_2hop (v7, off) | idx_graph_out | 2-hop forward expansion. Disabled — crashed in v7.2. | 1500 | 0.0 |
| 10 | concept_en | idx_concept_en | LLM concepts → English-tagged rows. **Token-overlap matcher** (stopwords filtered, weighted by shared meaningful tokens). | 3000 | 1.8 |
| 11 | term_orig | idx_term_orig + idx_term_lemma | LLM DE/FR terms → original-language-tagged rows. **German lemmatizer** (drop -en/-e/-er/-s with 4-char floor) + substring overlap ratio. | 2500 | 1.2 |
| 12 | bm25 | FTS5 per-language (de/fr/it/en) + enhance() | **Multi-language FTS5**: 4 separate indices, query in each language with appropriate term targets, merge by max-score. | 2000 | 0.8 |
| 13 | vector_raw | E_GPU brute-force | Dense semantic match on raw query. | 2000 | 1.0 |
| 14 | vector_enriched | E_GPU brute-force | Dense semantic match on keyword-enriched query. | 2000 | 1.0 |

### Guarantee channels (7) — round-robin merged, ~130 each = 910 slots + 90 RRF tail
1. `law_direct_match`, 2. `per_area_bedrock`, 3. `statute_backprop`,
4. `concept_en` (NEW), 5. `graph_forward` (NEW), 6. `sibling_expansion` (NEW), 7. `term_orig` (NEW)

### Role-aware negative gate
Substantive paragraph roles (`facts`, `reasoning`, `legal_standard`, `application`, `holding`, `citation`, `procedural_history`) override the noisy `is_notification_paragraph` enrichment flag.


# Phase 1 — Setup


In [1]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 1024**3:.1f} GB")
else:
    print("[WARN] No CUDA GPU available — vector channels will be skipped.")

Python: 3.12.13
PyTorch: 2.10.0+cu128
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, 95.0 GB


## 1.2 Mount Google Drive

**What:** Mount Drive so we can read the corpus, embeddings, and citation
graph from `/content/drive/MyDrive/swiss_law/`.

**Why:** The 21 GB of fp16 embeddings, 24 GB unified retrieval SQLite, and
2.4 GB citation graph all live on Drive — too large to download per run.

**Expected:** "Mounted at /content/drive". Skip silently if running locally.

**Failure modes:**
- "Drive not authorized" → click the OAuth link Colab prints.
- Mount succeeds but `MyDrive/swiss_law` is empty → wrong account; remount.

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"[skip] Not on Colab or drive already mounted: {e}")

Mounted at /content/drive


## 1.3 Resolve all data paths

**What:** Auto-detect `DATA_ROOT` (could be Drive, local, or Kaggle) and
build a `PATHS` dict pointing at every required artifact.

**Why:** Same notebook should run on Colab, Kaggle, or locally without code
changes. Each artifact is checked for existence with a clear OK/MISSING flag.

**Required inputs:**
- `data/val.csv` — 10 English queries with gold citations.
- `law_llm_descriptors_*.jsonl` — LLM enrichment of all 175k laws.
- `court_authority_cards_v5_unified.jsonl` — court enrichment (10.4 GB).
- `embeddings/qwen3_8b_unified_chunk*.npy` — 27 fp16 chunks (21 GB total).
- `embeddings/qwen3_8b_unified_manifest.parquet` — doc_id ↔ row_index.
- **v7 NEW:** `data_insights/citation_graph_extracted.sqlite` — 4-layer graph (~2.4 GB).

**Expected:** All flags `OK`. If `graph_db` is `MISSING`, graph channels
silently skip (recall drops back to v6 levels).

**Failure modes:**
- `MISSING law_llm` or `court_v5` → notebook can't build any index. Stop.
- `MISSING emb_dir` → vector channels skipped; BM25 + anchors still run.
- `MISSING graph_db` → graph channels skipped; sibling_expansion (court_base) only.

In [3]:
from pathlib import Path

CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/swiss_law"),
    Path("/content/drive/MyDrive/swiss_citation_extraction"),
    Path("/content/swiss_citation_extraction"),
    Path(r"E:/swiss_citation_extraction"),
    Path.cwd(),
]

DATA_ROOT = None
for root in CANDIDATE_ROOTS:
    if (root / "data" / "val.csv").exists():
        DATA_ROOT = root; break
if DATA_ROOT is None:
    print("[warn] Could not auto-detect DATA_ROOT — defaulting to /content/drive/MyDrive/swiss_law")
    DATA_ROOT = Path("/content/drive/MyDrive/swiss_law")
print(f"DATA_ROOT = {DATA_ROOT}")

def first_existing(*paths):
    for p in paths:
        if p.exists(): return p
    return paths[0]

PATHS = {
    "val_csv": DATA_ROOT / "data" / "val.csv",
    "law_llm": first_existing(
        DATA_ROOT / "data" / "checkpoints" / "law_llm_descriptors_0000000_all.jsonl",
        DATA_ROOT / "law_json_llm_output" / "law_llm_descriptors_0000000_all.jsonl",
    ),
    "court_v5": first_existing(
        DATA_ROOT / "artifacts_v2" / "court_authority_cards_v5_unified.jsonl",
        DATA_ROOT / "artifacts" / "court_authority_cards_v5_unified.jsonl",
    ),
    "emb_dir":      DATA_ROOT / "artifacts" / "embeddings",
    "emb_manifest": DATA_ROOT / "artifacts" / "embeddings" / "qwen3_8b_unified_manifest.parquet",
    "graph_db": first_existing(
        DATA_ROOT / "data_insights" / "citation_graph_extracted.sqlite",
        DATA_ROOT / "citation_graph_extracted.sqlite",
    ),
    "out_dir":      DATA_ROOT / "research" / "anchor_funnel_val001_v7",
}
PATHS["out_dir"].mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    if k == "out_dir": continue
    flag = "OK     " if p.exists() else "MISSING"
    print(f"  {flag}  {k:<14} {p}")

EMB_AVAILABLE   = PATHS["emb_dir"].exists() and any(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
GRAPH_AVAILABLE = PATHS["graph_db"].exists()
print()
print(f"  EMB_AVAILABLE   = {EMB_AVAILABLE}")
print(f"  GRAPH_AVAILABLE = {GRAPH_AVAILABLE}")

DATA_ROOT = /content/drive/MyDrive/swiss_law
  OK       val_csv        /content/drive/MyDrive/swiss_law/data/val.csv
  OK       law_llm        /content/drive/MyDrive/swiss_law/data/checkpoints/law_llm_descriptors_0000000_all.jsonl
  OK       court_v5       /content/drive/MyDrive/swiss_law/artifacts_v2/court_authority_cards_v5_unified.jsonl
  OK       emb_dir        /content/drive/MyDrive/swiss_law/artifacts/embeddings
  OK       emb_manifest   /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_unified_manifest.parquet
  OK       graph_db       /content/drive/MyDrive/swiss_law/data_insights/citation_graph_extracted.sqlite

  EMB_AVAILABLE   = True
  GRAPH_AVAILABLE = True


## 1.4 Knob panel (CONFIG)

**What:** All architecture knobs live in a single `CONFIG` dict — budgets,
RRF k, BM25 limits, vector top-k, query-expansion model, etc.

**Why:** Editing budgets without touching channel code is essential when
diagnosis suggests "channel X dropped this gold due to truncation, lift
budget". Every budget is rationalized in a comment.

**Sensitive knobs:**
- `budget_sibling = 2000` — was 500 in v6, dropped sibling Es non-deterministically.
- `budget_graph_forward = 1500` — graph 1-hop forward; case-level fan-out adds
  up to ~30 expansions per seed, 1500 covers 50 seeds × 30 siblings.
- `budget_graph_reverse = 1000` — co-citing rows; bounded to keep ranking signal.
- `enable_graph_2hop = False` — 2-hop tends to dump procedural articles already
  caught elsewhere; flip to True only after Phase 10 says so.

**Expected:** dict prints cleanly; nothing should be `None` except `budget_law_direct`.

In [4]:
import json

CONFIG = {
    "topk_final": 50000,

    # --- Channel budgets ---------------------------------------------------
    # law_direct_match: every law row whose canonical citation matches a
    # (LLM-named OR co-cited) statute target. Tiny per canon; uncapped is safe.
    "budget_law_direct":     None,

    # court_statute: court rows annotated with one of the LLM-named statutes.
    "budget_court_statute":  8000,   # v7.5: lifted 600->8000 (multi-match scoring + role boost now have room)

    # concept_en: rows whose concepts_en token overlaps with LLM concept_targets.
    "budget_concept":        3000,   # v7.5: lifted (rank-200+ gold lost otherwise)

    # term_orig: rows whose terms_original (DE/FR/IT) overlap with LLM term_targets.
    "budget_term":           2500,   # v7.5: lifted (deep-rank gold survives)

    # per_area_bedrock: most-cited canonical statutes within the LLM-named legal_area.
    # Per-area_top_n is internal cap; this budget caps the bedrock channel output.
    "budget_per_area":       1500,   # v7.5: procedural cluster lives at rank 100-500

    # co_citation: for each LLM statute target, fetch top-K co-citation neighbours
    # and pull their law + court rows.
    "budget_co_citation":    2500,   # v7.5: more co-cited candidates

    # bm25: lexical match on enriched query text. enhance() adds top-K corpus-
    # associated codes to query.
    "budget_bm25":           2000,   # v7.5: multi-language each gets 500

    # vector_raw / vector_enriched: dense semantic match using Qwen3-Embedding-8B
    # against full-corpus E_GPU.
    "budget_vector":         2000,   # v7.5: Obs 3 ceiling 0.289 — need deeper pool
    "budget_vector_enriched":2000,
    "budget_vector_hyde":    2000,   # HyDE — hypothetical-answer vector


    # statute_backprop: each caught court row contributes its cited statutes;
    # score = number of distinct caught court rows citing that article.
    # Surfaces procedural cluster (Art. 100 BGG, Art. 422 StPO, etc.).
    "budget_backprop":       2000,   # v7.5: rare-specificity scoring + more law candidates

    # sibling_expansion (court_base): caught court row → all Es of same judgment.
    # v6 had 500 → non-deterministic slice dropped val_001 sibling Es. 2000 fits
    # 100 seeds × 20 Es each.
    "budget_sibling":        5000,   # v7.5: judgment-importance scoring + bigger pool

    # graph_forward / graph_reverse / graph_2hop: 1-hop and 2-hop traversal of
    # the citation graph (intra-judgment backrefs + date aliases + range +
    # case-level fan-out). budget_forward sized for case-level fan-out from
    # ~50 seeds × 30 sibling-fanout ≈ 1500.
    "budget_graph_forward":  5000,   # v7.5: case-level fan-out + judgment importance
    "budget_graph_reverse":  3000,
    "enable_graph_2hop":     False,
    "budget_graph_2hop":     1500,

    # --- RRF + guarantee ---------------------------------------------------
    "rrf_k": 60,
    # Channels whose hits are PREPENDED before the RRF tail in Phase 9.
    # Keep this list small — graph_forward/reverse compete via RRF, only the
    # high-precision channels are guaranteed.
    # v7.5: 7-channel guarantee with smaller per-channel cap. Round-robin
    # ensures each high-recall channel contributes regardless of RRF score.
    # 7 channels × cap=130 = 910 guarantee slots, leaves ~90 RRF tail.
    "guarantee_channels": [
        "law_direct_match",
        "per_area_bedrock",
        "statute_backprop",
        "concept_en",            # v7.5 NEW
        "graph_forward",         # v7.5 NEW
        "sibling_expansion",     # v7.5 NEW
        "term_orig",             # v7.5 NEW
    ],
    "guarantee_per_channel": 130,
    # v7.5 weights — updated based on v7.4 per-channel mean recall + structural
    # value (multi-channel role).
    "channel_weights": {
        "statute_backprop":  2.5,   # mean recall 0.453 — universal best
        "graph_forward":     2.0,   # mean recall 0.324 — case-level fan-out
        "concept_en":        1.8,   # v7.5 lifted — token-overlap matcher now strong
        "court_statute":     1.5,   # v7.5 lifted — specificity+role scoring
        "per_area_bedrock":  1.5,
        "vector_raw":        1.0,
        "vector_enriched":   1.0,
        "vector_hyde":       1.5,   # HyDE — strongest vector signal expected

        "term_orig":         1.2,   # lemma+substring scoring
        "law_direct_match":  1.2,
        "sibling_expansion": 1.0,
        "bm25":              0.8,
        "co_citation":       0.7,
        "graph_reverse":     0.5,
        "graph_2hop":        0.0,
    },
    # v7.5 NEW: corpus-derived code-family expansion for per_area_bedrock.
    # When LLM names "StPO", we ALSO admit StBOG/BGG/BV/EMRK canons via
    # corpus co-citation evidence (not a hardcoded list).
    "code_family_top_k": 8,

    # --- Per-area bedrock --------------------------------------------------
    "per_area_top_n": 1000,   # v7.5: procedural cluster (422/428/135 StPO) lives at rank 100-500

    # --- Co-citation -------------------------------------------------------
    "co_citation_top_k_per_target":    50,   # v7.5: more cluster expansion
    "co_citation_min_co_count":        50,
    # drop neighbours that are TOO globally common (Art. 36 BV cited everywhere
    # would pollute court_statute). 5000 = ~0.2% of 2.5M corpus.
    "co_citation_max_neighbour_count": 50000,  # v7.5: allow more common; specificity downranks noise

    # --- Concept matching --------------------------------------------------
    "concept_substring_top_k": 6,

    # --- BM25 --------------------------------------------------------------
    "bm25_max_query_terms": 60,    # cap to stop token-explosion from enriched query
    "bm25_min_token_len":   3,

    # --- Vector ------------------------------------------------------------
    "vector_emb_model": "Qwen/Qwen3-Embedding-8B",
    "vector_topk":      800,

    # --- Query expansion ---------------------------------------------------
    "qwen_query_model":    "Qwen/Qwen3-32B",
    "qwen_max_new_tokens": 1024,

    # --- enhance() — corpus-derived BM25 lexicon expansion -----------------
    "enhance_top_k_codes":   5,
    "enhance_repeat_count":  5,
    "enhance_min_idf":       1.0,

    # --- Negative gate -----------------------------------------------------
    "noise_paragraph_roles": {"notification", "header", "empty", "metadata"},

    "lowercase_concepts": True,
    "lowercase_terms":    True,
}

print(json.dumps({k: v for k, v in CONFIG.items() if not isinstance(v, set)}, indent=2, default=str))

{
  "topk_final": 50000,
  "budget_law_direct": null,
  "budget_court_statute": 8000,
  "budget_concept": 3000,
  "budget_term": 2500,
  "budget_per_area": 1500,
  "budget_co_citation": 2500,
  "budget_bm25": 2000,
  "budget_vector": 2000,
  "budget_vector_enriched": 2000,
  "budget_vector_hyde": 2000,
  "budget_backprop": 2000,
  "budget_sibling": 5000,
  "budget_graph_forward": 5000,
  "budget_graph_reverse": 3000,
  "enable_graph_2hop": false,
  "budget_graph_2hop": 1500,
  "rrf_k": 60,
  "guarantee_channels": [
    "law_direct_match",
    "per_area_bedrock",
    "statute_backprop",
    "concept_en",
    "graph_forward",
    "sibling_expansion",
    "term_orig"
  ],
  "guarantee_per_channel": 130,
  "channel_weights": {
    "statute_backprop": 2.5,
    "graph_forward": 2.0,
    "concept_en": 1.8,
    "court_statute": 1.5,
    "per_area_bedrock": 1.5,
    "vector_raw": 1.0,
    "vector_enriched": 1.0,
    "vector_hyde": 1.5,
    "term_orig": 1.2,
    "law_direct_match": 1.2,
    "sib

## 1.5 Load all queries from val.csv

No per-query branching: every row in `val.csv` is processed by the same code path.


In [5]:
import pandas as pd

val_df = pd.read_csv(PATHS["val_csv"])
print(f"val.csv has {len(val_df)} queries\n")

ALL_QUERIES = []
for _, row in val_df.iterrows():
    qid = str(row["query_id"])
    qtext = str(row["query"])
    gold = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]
    ALL_QUERIES.append({"query_id": qid, "query_text": qtext, "gold": gold})

for q in ALL_QUERIES:
    head = q["query_text"][:90] + ("..." if len(q["query_text"]) > 90 else "")
    print(f"  {q['query_id']:<8}  gold={len(q['gold']):>2}   {head}")


val.csv has 10 queries

  val_001   gold=42   May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 A...
  val_002   gold=36   A claimant holding a national vocational diploma in warehouse operations worked intermitte...
  val_003   gold=47   A. Rivera, a Peruvian national born in 1994 and with no prior convictions in the forum sta...
  val_004   gold=10   Mr. Dalton, born in 1941 and resident in a small lakeside town near Thun, executed a handw...
  val_005   gold=11   A parent, separated from their co-parent since 2008, has not had custody of the two childr...
  val_006   gold=18   On 3 March 2012, homeowners Ms. L and her partner Mr. M asked G, an installer they knew so...
  val_007   gold=19   An heirship claims title to a vintage pocket chronometer known as “The Meridian” that belo...
  val_008   gold=29   Has a member of the town council of the Borough of L., who chaired the board of a publicly...
  val_009   gold=14   A divorced custodial paren

# Phase 2 — Build all corpus indexes (one pass)

## 2.1 What this big cell does

This cell is the heart of the retrieval pipeline. It **streams both JSONL files**
(`law_llm_descriptors` + `court_authority_cards_v5`) and builds **all** in-memory
indexes the channels need:

| Index | Type | What it maps |
|---|---|---|
| `cit_to_doc_ids[citation]` | dict[str, list[str]] | citation string → list of doc_ids |
| `doc_meta[did]` | dict[str, dict] | doc_id → {citation, family, court_base, paragraph_role} |
| `idx_law_direct[canonical]` | dict[str, set] | "100 BGG" → law doc_ids |
| `idx_court_statute[canonical]` | dict[str, set] | "100 BGG" → court rows whose anchors include this |
| `idx_court_base[base]` | dict[str, set] | "137 IV 122" → all Es of judgment (used by sibling_expansion) |
| `idx_concept_en[token]` | dict[str, set] | English concept → doc_ids |
| `idx_term_orig[token]` | dict[str, set] | DE/FR/IT term → doc_ids |
| `search_text[did]` | dict[str, str] | doc_id → BM25-search text (concatenated enrichment) |
| `legal_area_per_doc[did]` | dict[str, str] | court doc → its `legal_area_static` |
| `co_citation_pairs` | Counter[(canon_a, canon_b)] | unordered pairs of statutes co-cited within same court row |
| `tlf[token][code]` | dict[str, Counter] | corpus-wide token → law-code association (for `enhance()`) |
| `doc_statute_anchors[did]` | dict[str, set] | court doc_id → set of canonical statutes it cites |

## 2.2 Why we need each one (channel attribution)

- `idx_law_direct` → channels: `law_direct_match`, `per_area_bedrock`, `statute_backprop`, `co_citation`
- `idx_court_statute` → channels: `court_statute`, `co_citation`
- `idx_court_base` → channel: `sibling_expansion` (own-judgment Es)
- `idx_concept_en` / `idx_term_orig` → channels: `concept_en`, `term_orig`
- `search_text` → channel: `bm25` (FTS5 indexed in Phase 5)
- `co_citation_pairs` → Phase 4 builds `co_neighbours` from this; channel: `co_citation`
- `tlf` → BM25 query enhancement in Phase 5
- `doc_statute_anchors` → channel: `statute_backprop`

## 2.3 Expected outputs
- Law: ~175k rows in ~10 s
- Court: ~2.47M rows in ~150 s
- Token→code association: ~90k tokens, avg ~10 codes/token
- Co-citation pairs: ~1.1M

## 2.4 Failure modes
- **Slower than 200 s for court** → Drive throttling; retry.
- **`Total docs ≠ ~2.65M`** → JSONL truncation; check file size matches local.
- **`tlf` very small (< 50k tokens)** → law tokenizer pattern wrong; check the
  regex split below.

In [6]:
from collections import defaultdict, Counter
import re, json, time

# --- Statute / case canonicalizers --------------------------------------------
CODE_ALIAS = {
    "CPP": "StPO", "CP": "StGB", "CC": "ZGB", "CO": "OR",
    "LTF": "BGG", "LACI": "AVIG", "LAA": "UVG",
    "LP": "SchKG", "LDIP": "IPRG", "Cst": "BV", "Cst.": "BV",
    "STPO": "StPO", "OBG": "OR",
}
ART_RE = re.compile(r"art\.?\s*(\d+[a-z]?)", re.I)
CODE_RE = re.compile(r"\b([A-Z][A-Za-z]{1,8}\.?)\b")

def statute_anchor_canonical(raw):
    if not raw: return None
    s = raw.strip()
    m = ART_RE.search(s)
    if not m: return None
    cands = [c.strip(".") for c in CODE_RE.findall(s)
             if c.strip(".") not in ("Art","Abs","Ziff","lit","let","al","Bst")]
    if not cands: return None
    code = CODE_ALIAS.get(cands[-1], cands[-1])
    return f"{m.group(1)} {code}"

def article_num(raw):
    if not raw: return None
    m = ART_RE.search(raw.strip())
    return m.group(1) if m else None

LEGAL_AREA_DEFAULT_CODE = {
    "criminal law and criminal procedure": "StPO",
    "criminal procedure":                  "StPO",
    "criminal law":                        "StGB",
    "civil law":                           "ZGB",
    "obligations":                         "OR",
    "civil procedure":                     "ZPO",
    "constitutional and public law":       "BV",
    "constitutional law":                  "BV",
    "administrative law":                  "VwVG",
    "social insurance":                    "ATSG",
    "tax law":                             "DBG",
}

def canonicalize_row_anchors(raw_anchors, legal_area_static):
    canons = set()
    primary_code = None
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            primary_code = c.split()[1]; break
    fallback = primary_code
    if fallback is None and legal_area_static:
        la = legal_area_static.lower()
        for k, v in LEGAL_AREA_DEFAULT_CODE.items():
            if k in la:
                fallback = v; break
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            canons.add(c); continue
        n = article_num(sa)
        if n and fallback:
            canons.add(f"{n} {fallback}")
    return canons

CASE_BGE_RE    = re.compile(r"BGE\s+(\d+)\s+([IVX]+)\s+(\d+)")
CASE_DOCKET_RE = re.compile(r"\b(\d[A-Z]_\d+/\d{4})\b")

def case_anchor_canonical(raw):
    if not raw: return None
    s = raw.strip()
    m = CASE_BGE_RE.search(s)
    if m: return f"BGE {m.group(1)} {m.group(2)} {m.group(3)}"
    m = CASE_DOCKET_RE.search(s)
    if m: return m.group(1)
    return None

TOKEN_NORM_RE = re.compile(r"\s+")
def norm_token(s, lower):
    if not s: return None
    s = TOKEN_NORM_RE.sub(" ", s.strip())
    if not s: return None
    return s.lower() if lower else s

# --- German lemmatizer for term_orig channel ---------------------------------
# Drop one common inflectional suffix at a time. Only drop a suffix if the
# remaining stem is >= 4 chars; allow recursion (e.g. 'haftens' -> 'haften'
# -> 'haft'). Surface form is always kept in idx_term_orig as well, so this
# is purely additive.
_TERM_LEMMA_SUFFIXES = ("en", "es", "em", "er", "e", "n", "s")
def term_lemma(tok):
    if not tok: return tok
    cur = tok
    seen = {cur}
    while True:
        changed = False
        for suf in _TERM_LEMMA_SUFFIXES:
            if cur.endswith(suf) and len(cur) - len(suf) >= 4:
                stem = cur[: len(cur) - len(suf)]
                if stem not in seen:
                    cur = stem; seen.add(cur); changed = True; break
        if not changed:
            break
    return cur

# --- Indexes -----------------------------------------------------------------
cit_to_doc_ids       = defaultdict(list)
doc_meta             = {}
idx_law_direct       = defaultdict(set)
idx_court_statute    = defaultdict(set)
idx_case_anchor      = defaultdict(set)
idx_court_base       = defaultdict(set)
idx_concept_en       = defaultdict(set)
idx_term_orig        = defaultdict(set)
idx_term_lemma       = defaultdict(set)   # lemma-form -> set(doc_ids)
term_orig_keys       = set()              # all normalized surface-form keys (for substring scan)
legal_area_per_doc   = {}
search_text          = {}
co_citation_pairs    = Counter()
tlf                  = defaultdict(Counter)
token_doc_count      = Counter()
doc_statute_anchors  = {}
doc_language         = {}

DOC_ID_LAW   = lambda i: f"law:{i}"
DOC_ID_COURT = lambda i: f"court:{i}"

def _take_text(*parts, max_chars=2000):
    out = []
    for p in parts:
        if not p: continue
        if isinstance(p, list):
            for x in p:
                if isinstance(x, str): out.append(x)
                elif isinstance(x, dict):
                    for v in x.values():
                        if isinstance(v, str): out.append(v)
        elif isinstance(p, str):
            out.append(p)
    return (" ".join(out))[:max_chars]

# --- Stream law jsonl --------------------------------------------------------
t0 = time.time(); n_law = 0
with open(PATHS["law_llm"], encoding="utf-8") as f:
    for line in f:
        try: obj = json.loads(line)
        except Exception: continue
        cit = obj.get("citation","")
        if not cit: continue
        did = DOC_ID_LAW(n_law)
        cit_to_doc_ids[cit].append(did)
        doc_meta[did] = {"citation": cit, "family": "law", "court_base": None,
                         "paragraph_role": None, "is_notification_paragraph": False}
        canon = statute_anchor_canonical(cit)
        if canon: idx_law_direct[canon].add(did)

        enr = obj.get("llm_enrichment") or {}
        terms_de = []; terms_en = []
        for t in enr.get("terms_de_to_en") or []:
            if isinstance(t, dict):
                de = norm_token(t.get("de",""), CONFIG["lowercase_terms"])
                en = norm_token(t.get("en",""), CONFIG["lowercase_terms"])
                if de:
                    idx_term_orig[de].add(did); terms_de.append(de)
                    term_orig_keys.add(de)
                    _lem_de = term_lemma(de)
                    if _lem_de and _lem_de != de: idx_term_lemma[_lem_de].add(did)
                    idx_term_lemma[de].add(did)
                if en:
                    idx_concept_en[en].add(did); terms_en.append(en)
        for c in enr.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        search_text[did] = _take_text(
            cit, enr.get("english_summary",""), enr.get("legal_rule",""),
            enr.get("legal_question",""), enr.get("applicability_conditions"),
            enr.get("concepts_en"), terms_de, terms_en,
        )
        legal_area_per_doc[did] = "law"
        doc_language[did] = (obj.get("language") or "de").lower()

        # token -> code association (only law rows have a clean canonical code).
        if canon and " " in canon:
            row_code = canon.split()[1].lower()
            row_text = search_text[did].lower()
            row_tokens = set()
            for tok in re.split(r"[^\w\d]+", row_text, flags=re.UNICODE):
                if len(tok) >= 3:
                    row_tokens.add(tok)
            for tok in row_tokens:
                tlf[tok][row_code] += 1
                token_doc_count[tok] += 1

        n_law += 1

print(f"Law: {n_law:,} rows indexed in {time.time()-t0:.1f}s")
print(f"Token->code association: {len(tlf):,} tokens, "
      f"avg codes/token = {sum(len(c) for c in tlf.values())/max(1,len(tlf)):.1f}")

# --- Stream court jsonl ------------------------------------------------------
t1 = time.time(); n_court = 0
with open(PATHS["court_v5"], encoding="utf-8") as f:
    for line in f:
        try: obj = json.loads(line)
        except Exception: continue
        cit = obj.get("citation","")
        if not cit: continue
        did = DOC_ID_COURT(n_court)
        cit_to_doc_ids[cit].append(did)
        cb  = obj.get("court_base") or ""
        rag = obj.get("rag_enrichment") or {}
        legal_area_static = obj.get("legal_area_static") or rag.get("legal_area") or ""
        doc_meta[did] = {"citation": cit, "family": "court", "court_base": cb,
                         "paragraph_role": rag.get("paragraph_role"),
                         "is_notification_paragraph": bool(obj.get("is_notification_paragraph"))}
        legal_area_per_doc[did] = (legal_area_static or "").lower()
        _row_lang_raw = (obj.get("language") or "").lower()
        doc_language[did] = _row_lang_raw if _row_lang_raw in ("de", "fr", "it", "en") else "en"

        if cb:
            idx_court_base[cb].add(did)
            cb_canon = case_anchor_canonical(cb)
            if cb_canon: idx_case_anchor[cb_canon].add(did)

        row_canons = canonicalize_row_anchors(rag.get("statute_anchors") or [], legal_area_static)
        for canon in row_canons:
            idx_court_statute[canon].add(did)
        if row_canons:
            doc_statute_anchors[did] = row_canons
        rc = sorted(row_canons)
        for i in range(len(rc)):
            for j in range(i+1, len(rc)):
                co_citation_pairs[(rc[i], rc[j])] += 1

        for ca in rag.get("case_anchors") or []:
            canon = case_anchor_canonical(ca)
            if canon: idx_case_anchor[canon].add(did)
        for c in rag.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        for t in rag.get("terms_original") or []:
            tok = norm_token(t, CONFIG["lowercase_terms"])
            if tok:
                idx_term_orig[tok].add(did)
                term_orig_keys.add(tok)
                _lem = term_lemma(tok)
                if _lem and _lem != tok: idx_term_lemma[_lem].add(did)
                idx_term_lemma[tok].add(did)

        search_text[did] = _take_text(
            cit, obj.get("text_excerpt_original",""),
            rag.get("concepts_en"), rag.get("terms_original"),
            rag.get("micro_topic",""), rag.get("topic",""), rag.get("subtopic",""),
            rag.get("statute_anchors"),
        )

        n_court += 1
        if n_court % 500_000 == 0:
            print(f"  court progress: {n_court:,} rows ({time.time()-t1:.1f}s)")

print(f"Court: {n_court:,} rows indexed in {time.time()-t1:.1f}s")
# Pre-compute per-canon document counts for specificity weighting in channel_court_statute.
idx_court_statute_count = {canon: len(s) for canon, s in idx_court_statute.items()}
print(f"Total docs:        {len(doc_meta):,}")
print(f"Unique citations:  {len(cit_to_doc_ids):,}")
print(f"Index sizes:       law_direct={len(idx_law_direct):,}, court_statute={len(idx_court_statute):,}, "
      f"case={len(idx_case_anchor):,}, court_base={len(idx_court_base):,}, "
      f"concept={len(idx_concept_en):,}, term={len(idx_term_orig):,}, "
      f"term_lemma={len(idx_term_lemma):,}, term_keys={len(term_orig_keys):,}")
print(f"Co-citation pairs: {len(co_citation_pairs):,}")

Law: 173,033 rows indexed in 11.2s
Token->code association: 91,173 tokens, avg codes/token = 11.7
  court progress: 500,000 rows (36.3s)
  court progress: 1,000,000 rows (78.5s)
  court progress: 1,500,000 rows (117.0s)
  court progress: 2,000,000 rows (156.7s)
Court: 2,476,315 rows indexed in 192.9s
Total docs:        2,649,348
Unique citations:  2,158,211
Index sizes:       law_direct=49,288, court_statute=81,988, case=157,227, court_base=178,593, concept=292,946, term=363,331, term_lemma=521,146, term_keys=363,331
Co-citation pairs: 1,142,790


## 2.5 Map gold citations to doc_ids (per query)

For each query we resolve its gold citations against the corpus. This is the upper bound on R@K — anything not mapped here cannot be retrieved by any channel.


In [7]:
ALL_GOLD_DOC_SET = {}
ALL_TOTAL_GOLD = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    gset = set()
    unmapped = []
    for g in q["gold"]:
        dids = cit_to_doc_ids.get(g, [])
        if not dids:
            unmapped.append(g)
        else:
            gset.update(dids)
    ALL_GOLD_DOC_SET[qid] = gset
    ALL_TOTAL_GOLD[qid] = len(q["gold"])
    warn = f"  [WARN unmapped={len(unmapped)}]" if unmapped else ""
    print(f"  {qid:<8}  gold={len(q['gold']):>2}  mapped={len(q['gold']) - len(unmapped):>2}  doc_ids={len(gset):>4}{warn}")


  val_001   gold=42  mapped=42  doc_ids=  42
  val_002   gold=36  mapped=36  doc_ids=  38
  val_003   gold=47  mapped=47  doc_ids=  47
  val_004   gold=10  mapped=10  doc_ids=  10
  val_005   gold=11  mapped=11  doc_ids=  11
  val_006   gold=18  mapped=18  doc_ids=  18
  val_007   gold=19  mapped=19  doc_ids=  19
  val_008   gold=29  mapped=29  doc_ids=  30
  val_009   gold=14  mapped=14  doc_ids=  14
  val_010   gold=25  mapped=25  doc_ids=  25


# Phase 3 — Citation graph (v7 NEW)

## 3.1 What this cell does

Loads `data_insights/citation_graph_extracted.sqlite` (built locally via 4
alias passes; ~2.4 GB; 23.65 M edges) and converts edge tuples (citation_str
→ citation_str) into doc_id-keyed in-memory dicts:
- `idx_graph_out[did]` → list of doc_ids the row text-cites or fans-out to
- `idx_graph_in[did]` → list of doc_ids whose text cites this row

## 3.2 Why we need this — the 4 layers

The graph encodes signal that's **invisible to BM25, vector, and concepts**:

1. **Intra-judgment back-references**
   - Court text uses bare back-refs like `(vgl. E. 6.2 hiervor)` — references
     to siblings of the same judgment. Original `extract_citation_graph.py`
     missed all of these (its CONSIDERATION_RE only fires after a docket).
   - Pass 1 (`extract_intra_judgment_backrefs.py`) handles 4 patterns × 3
     languages: `vgl./siehe E. N`, `E. N hiervor`, `E. N ci-dessus`, `cf. supra
     consid. N`, including `siehe oben E. N`, `hiervor E. N`, `vorstehend`.
   - Self-tested with 15 real-world cases; 4-layer audit on 500 marker-rows
     dropped zero-target rate from 335→175 (90% of remaining are TRUE false
     markers like `nach oben` = "to the top").

2. **Date-stripped aliases**
   - Corpus extraction stores dated form `1B_210/2023 12.05.2023 E. 3`, but
     val gold uses un-dated form `1B_210/2023 E. 3`.
   - Pass 2 adds alias edges from dated → un-dated forms (only when the
     un-dated form exists as a real corpus row).

3. **E.-range expansion**
   - Corpus stores ranges as one citation: `1B_90/2021 E. 2.1-2.4`. Gold uses
     individual Es: `E. 2.1`, `E. 2.2`, `E. 2.3`, `E. 2.4`.
   - Pass 3 enumerates ranges and adds aliases.

4. **Case-level fan-out**
   - When a source cites one E. of a judgment, gold may include OTHER Es of
     the same judgment that nobody text-cites by exact pinpoint. Treats
     "citing one E." as "this case is relevant".
   - Pass 4 fans out: edge → BASE E. X spawns alias edges to all `BASE E. Y`
     where Y exists as a real corpus row.
   - Largest pass: +18.77 M edges.

After all 4 passes: **0/42 val_001 gold orphan** (was 11 originally).

## 3.3 Why doc_id mapping skips synthetic targets

Some graph nodes are synthetic (e.g., a backref `E. 4 hiervor` resolves to
`{base} E. 4` which may not be a real corpus row). Such targets have no
`cit_to_doc_ids` entry and are skipped here — graph channels only retrieve
real corpus rows.

## 3.4 Expected outputs
- ~24 M edges loaded (some skipped because synthetic targets have no doc_id)
- Out-degree avg ~10–15 (reflects case-level fan-out per cited judgment)
- In-degree avg ~10–15
- Load time: 30–60 s

## 3.5 Failure modes
- `graph_db MISSING` → graph channels skipped; sibling_expansion still works.
- `0 edges loaded` → all citations failed to map; check `cit_to_doc_ids`
  was built before this cell, and citations strings are exact (case, spacing).
- `Out-degree avg < 3` → mapping mostly failed; check date format normalization.

In [8]:
import sqlite3 as _sqlite3

idx_graph_out = defaultdict(list)
idx_graph_in  = defaultdict(list)
GRAPH_OK = bool(GRAPH_AVAILABLE)

if GRAPH_OK:
    _t = time.time()
    _cit_to_did = {cit: dids[0] for cit, dids in cit_to_doc_ids.items() if dids}
    print(f"Built citation->doc_id map ({len(_cit_to_did):,} entries)")

    _g = _sqlite3.connect(str(PATHS["graph_db"]))
    n_loaded = 0; n_skipped = 0
    for _src, _tgt in _g.execute(
        "SELECT source, target FROM edges WHERE dataset='court_considerations'"
    ):
        _s = _cit_to_did.get(_src)
        _t2 = _cit_to_did.get(_tgt)
        if _s is None or _t2 is None:
            n_skipped += 1; continue
        if _s == _t2: continue
        idx_graph_out[_s].append(_t2)
        idx_graph_in[_t2].append(_s)
        n_loaded += 1
    _g.close()
    print(f"Graph: {n_loaded:,} edges loaded, {n_skipped:,} skipped (cit not in corpus)")
    print(f"Graph: out-degree avg = {n_loaded/max(1,len(idx_graph_out)):.1f}, "
          f"in-degree avg = {n_loaded/max(1,len(idx_graph_in)):.1f}")
    print(f"Graph: load time {time.time()-_t:.1f}s")
else:
    print("[skip] Graph DB missing — graph channels will return empty lists.")

# --- Judgment-importance index (v7.4) -------------------------------------
# importance(court_base) = sum over did in idx_court_base[court_base] of
#                         len(idx_graph_in[did])
# i.e. total incoming citations across every E.-paragraph row of the
# judgment. Landmark BGE cases score thousands; obscure dockets score 0-50.
# Used by sibling_expansion / graph_forward / graph_reverse channels in
# cell 32 to break score=1 ties and prioritize widely-cited judgments.
idx_judgment_importance = {}
if GRAPH_OK:
    _ti = time.time()
    for _cb, _dids in idx_court_base.items():
        _s = 0
        for _d in _dids:
            _s += len(idx_graph_in.get(_d, ()))
        idx_judgment_importance[_cb] = _s
    _imp_vals = list(idx_judgment_importance.values())
    if _imp_vals:
        _imp_vals_sorted = sorted(_imp_vals)
        _n = len(_imp_vals_sorted)
        print(f"Judgment importance: {len(idx_judgment_importance):,} judgments, "
              f"median={_imp_vals_sorted[_n//2]}, "
              f"p75={_imp_vals_sorted[3*_n//4]}, "
              f"max={_imp_vals_sorted[-1]}")
    print(f"Judgment importance: built in {time.time()-_ti:.1f}s")
else:
    print("[skip] idx_judgment_importance empty — graph not loaded.")

Built citation->doc_id map (2,158,211 entries)
Graph: 20,494,436 edges loaded, 3,155,263 skipped (cit not in corpus)
Graph: out-degree avg = 27.2, in-degree avg = 14.0
Graph: load time 108.1s
Judgment importance: 178,593 judgments, median=7, p75=44, max=204686
Judgment importance: built in 1.1s


# Phase 4 — Per-area bedrock + co-citation neighbours

## 4.1 Per-area bedrock — why we need it

For a query in legal area "criminal procedure", certain articles are
**universally cited** by every BGer detention decision (Art. 100 BGG,
Art. 42 BGG, Art. 66 BGG — the procedural/cost cluster). The LLM rarely
names these because they're "implicit" to lawyers but they're often gold.

Per-area bedrock is **corpus-derived**: from `legal_area_per_doc`, count
which canonical statutes appear most often in court rows of each area.

**Filter:** v6 added a critical fix — restrict per-area bedrock to canonicals
whose code matches one of the LLM-named codes (e.g., StPO + BGG for val_001).
v4 returned BGG-dominated lists across ALL areas because BGG appeal articles
are cited everywhere.

**Expected:** ~26 distinct legal areas; `criminal procedure and coercive measures`
top-8 should include 66 BGG, 78 BGG, 81 BGG.

**Failure modes:**
- `0 areas` → `legal_area_per_doc` is empty; check court enrichment has
  `legal_area_static` set.

In [9]:
print("Building per-area bedrock index...")
_t = time.time()
per_area_canon_count = defaultdict(Counter)
for did, area in legal_area_per_doc.items():
    if not area or area == "law": continue
    canons = doc_statute_anchors.get(did, ())
    for canon in canons:
        per_area_canon_count[area][canon] += 1
print(f"  built in {time.time()-_t:.1f}s; areas: {len(per_area_canon_count)}")
for area in list(per_area_canon_count.keys())[:4]:
    print(f"  area={area!r}: top 8 = {per_area_canon_count[area].most_common(8)}")

# v7.5 NEW: derive code-family from corpus co-citation. For each pair of
# canons (a, b) co-cited in a court row, record the (code_a, code_b) pair
# weight. Used at retrieval time to expand statute_target_codes from
# LLM-named codes to corpus-related codes (no hardcoded statute cluster).
print("Building corpus-derived code-pair statistics...")
_tc = time.time()
code_pair_count = Counter()
for (a, b), n in co_citation_pairs.items():
    ca = a.split()[1] if " " in a else None
    cb = b.split()[1] if " " in b else None
    if ca and cb and ca != cb:
        code_pair_count[(ca, cb)] += n
        code_pair_count[(cb, ca)] += n   # symmetric
print(f"  code-pair statistics: {len(code_pair_count):,} directed pairs ({time.time()-_tc:.1f}s)")
print(f"  StPO's top related codes: {[(c, n) for (a, c), n in code_pair_count.most_common(2000) if a=='StPO'][:8]}")


Building per-area bedrock index...
  built in 2.1s; areas: 26
  area='constitutional and public law': top 8 = [('66 BGG', 13212), ('29 BV', 12189), ('89 BGG', 11479), ('82 BGG', 11464), ('42 BGG', 10836), ('9 BV', 9650), ('106 BGG', 9625), ('68 BGG', 8649)]
  area='administrative, tax, migration, and regulatory law': top 8 = [('42 BGG', 20301), ('106 BGG', 19460), ('66 BGG', 17471), ('105 BGG', 16033), ('95 BGG', 15722), ('83 BGG', 15231), ('68 BGG', 14674), ('89 BGG', 12187)]
  area='civil law': top 8 = [('63 OJ', 3189), ('8 ZGB', 2740), ('55 OJ', 2518), ('64 OJ', 2235), ('55 OG', 2085), ('63 OG', 1906), ('159 OG', 1876), ('9 BV', 1875)]
  area='criminal law and criminal procedure': top 8 = [('66 BGG', 28149), ('42 BGG', 20746), ('106 BGG', 19537), ('64 BGG', 13843), ('108 BGG', 12101), ('97 BGG', 12075), ('105 BGG', 11759), ('81 BGG', 11102)]
Building corpus-derived code-pair statistics...
  code-pair statistics: 54,790 directed pairs (0.7s)
  StPO's top related codes: [('BGG', 50026

## 4.2 Co-citation neighbours — why we need it

For each statute target the LLM names (e.g., `Art. 221 StPO`), find the top-K
canonical statutes that are **cited together** in the same court rows most
often. Surfaces statute clusters that move together in legal practice.

**Filter:** drop neighbours that are TOO globally common (Art. 36 BV cited
in nearly every criminal case). 5000 = ~0.2% of 2.5 M corpus.

**Expected:** for `221 StPO` neighbours: `212 StPO`, `237 StPO`, `5 StPO`,
`5 EMRK` (the detention statute cluster). NOT `36 BV` (filtered out).

**Failure modes:**
- All-empty neighbours → `co_citation_pairs` was empty; check court enrichment
  contained statute_anchors.

In [10]:
co_neighbours = defaultdict(list)
canon_count = Counter()
for did, canons in doc_statute_anchors.items():
    for c in canons: canon_count[c] += 1

for (a, b), cnt in co_citation_pairs.items():
    if cnt < CONFIG["co_citation_min_co_count"]: continue
    if canon_count[b] > CONFIG["co_citation_max_neighbour_count"]: pass  # may filter b
    if canon_count[a] > CONFIG["co_citation_max_neighbour_count"]: pass  # may filter a
    co_neighbours[a].append((b, cnt))
    co_neighbours[b].append((a, cnt))

# Apply frequency filter on neighbour side and keep top-K per source
co_neighbours = {
    src: sorted(
        ((nb, n) for nb, n in nbs if canon_count[nb] <= CONFIG["co_citation_max_neighbour_count"]),
        key=lambda x: -x[1]
    )[:CONFIG["co_citation_top_k_per_target"] * 2]
    for src, nbs in co_neighbours.items()
}
print(f"Co-citation neighbours indexed for {len(co_neighbours):,} canonicals "
      f"(after frequency filter: max global count = {CONFIG['co_citation_max_neighbour_count']}).")
print("Sample - neighbours of '221 StPO' AFTER frequency filter:")
for nb, n in co_neighbours.get("221 StPO", [])[:8]:
    print(f"  {nb}: co={n}, total_in_corpus={canon_count[nb]}")

Co-citation neighbours indexed for 2,282 canonicals (after frequency filter: max global count = 50000).
Sample - neighbours of '221 StPO' AFTER frequency filter:
  36 BV: co=915, total_in_corpus=7216
  31 BV: co=783, total_in_corpus=4339
  10 BV: co=731, total_in_corpus=5241
  212 StPO: co=714, total_in_corpus=1337
  221 BV: co=646, total_in_corpus=670
  237 StPO: co=487, total_in_corpus=1467
  5 BV: co=393, total_in_corpus=9685
  5 StPO: co=272, total_in_corpus=1955


# Phase 5 — BM25 (FTS5 in-memory)

## 5.1 What this cell does

Build SQLite FTS5 over `search_text[did]` for all 2.65 M docs. Provides
`bm25_search(query_text, k)` which returns top-k doc_ids ranked by BM25.

## 5.2 Why FTS5 (and not Whoosh / Lucene)

- Pure stdlib; no extra install.
- ~80 s build for 2.6 M short docs.
- Returns BM25-scored top-K in <50 ms.
- We can pass any expanded query text and get a stable ranking.

## 5.3 enhance() — corpus-derived query enrichment

Untitled75's reference notebook used a `enhance()` from train data that we
forbid. We replicate the IDEA (boost query with code names most associated
with query tokens) but train it on the **corpus** (laws_de) instead of train.

For each token in query, look up `tlf[token]` (a Counter mapping legal codes
to row counts). The top-K codes with highest score get appended to the query
multiple times, biasing BM25 toward law rows of those codes.

Example: query contains "detention" → boost `stpo` (high) more than `or`.

**Expected:** FTS5 build ~80 s. enhance() boost typically adds 5×5=25 token
repetitions to the query.

**Failure modes:**
- Slow build (>180 s) → swap MEMORY journal for OFF, or use a temp file.

In [11]:
import sqlite3, math

# -----------------------------------------------------------------------------
# Phase 5.A — legacy single-language FTS (kept for back-compat / debugging)
# -----------------------------------------------------------------------------
print(f"Building in-memory FTS5 (legacy, single index) over {len(search_text):,} docs...")
_t = time.time()
_fts = sqlite3.connect(":memory:")
_fts.execute("PRAGMA journal_mode = MEMORY")
_fts.execute("PRAGMA synchronous = OFF")
_fts.execute("CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, tokenize = 'unicode61 remove_diacritics 2')")
_inserted = 0
_batch = []
for did, txt in search_text.items():
    _batch.append((did, txt))
    if len(_batch) >= 50000:
        _fts.executemany("INSERT INTO docs(did, body) VALUES (?, ?)", _batch)
        _inserted += len(_batch); _batch.clear()
        if _inserted % 500000 == 0:
            print(f"  inserted {_inserted:,} ({time.time()-_t:.1f}s)")
if _batch:
    _fts.executemany("INSERT INTO docs(did, body) VALUES (?, ?)", _batch)
    _inserted += len(_batch)
_fts.commit()
print(f"FTS5 (legacy) built: {_inserted:,} rows in {time.time()-_t:.1f}s")


# -----------------------------------------------------------------------------
# Phase 5.B — per-language FTS5 indices
# -----------------------------------------------------------------------------
# Build one in-memory FTS5 per language. doc_language[did] was populated in
# cell 12 from the top-level `language` field (de/fr/it; anything else,
# including 'unknown' and missing, was normalised to 'en').
print("Building per-language FTS5 indices (de/fr/it/en)...")
_t_ml = time.time()
_LANGS = ("de", "fr", "it", "en")
_fts_by_lang = {}
_lang_doc_count = {}
for _L in _LANGS:
    _conn = sqlite3.connect(":memory:")
    _conn.execute("PRAGMA journal_mode = MEMORY")
    _conn.execute("PRAGMA synchronous = OFF")
    _conn.execute(
        "CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, "
        "tokenize = 'unicode61 remove_diacritics 2')"
    )
    _fts_by_lang[_L] = _conn
    _lang_doc_count[_L] = 0

# Group inserts by language. Stream search_text once, route per doc_language.
_batches = {L: [] for L in _LANGS}
_unknown_lang_did = 0
for did, txt in search_text.items():
    L = doc_language.get(did, "en")
    if L not in _fts_by_lang:
        # safety net for any unexpected value (shouldn't happen after cell 12)
        L = "en"
        _unknown_lang_did += 1
    _batches[L].append((did, txt))
    _lang_doc_count[L] += 1
    if len(_batches[L]) >= 50000:
        _fts_by_lang[L].executemany(
            "INSERT INTO docs(did, body) VALUES (?, ?)", _batches[L]
        )
        _batches[L].clear()

for L in _LANGS:
    if _batches[L]:
        _fts_by_lang[L].executemany(
            "INSERT INTO docs(did, body) VALUES (?, ?)", _batches[L]
        )
        _batches[L].clear()
    _fts_by_lang[L].commit()

print(f"Per-language FTS5 built in {time.time()-_t_ml:.1f}s")
for _L in _LANGS:
    print(f"  fts[{_L}]: {_lang_doc_count[_L]:,} docs")
if _unknown_lang_did:
    print(f"  (note) {_unknown_lang_did:,} docs had no language tag and were "
          f"routed to 'en'")


# -----------------------------------------------------------------------------
# Phase 5.C — enhance() (corpus-derived BM25 lexicon expansion); unchanged.
# -----------------------------------------------------------------------------
def _enhance_codes(text):
    text_lc = text.lower()
    tokens = set()
    for tok in re.split(r"[^\w\d]+", text_lc, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            tokens.add(tok)
    code_score = Counter()
    for tok in tokens:
        if tok not in tlf: continue
        n_docs = max(1, token_doc_count[tok])
        idf = math.log(1 + (max(1, len(search_text)) / n_docs))
        if idf < CONFIG["enhance_min_idf"]: continue
        for code, cnt in tlf[tok].most_common():
            code_score[code] += cnt * idf
    return [c for c, _ in code_score.most_common(CONFIG["enhance_top_k_codes"])]


# -----------------------------------------------------------------------------
# Phase 5.D — legacy bm25_search() (unchanged behaviour, single-index).
# -----------------------------------------------------------------------------
def bm25_search(query_text, k):
    boosted = _enhance_codes(query_text)
    enriched = query_text + " " + " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)
    fts_q = []
    for tok in re.split(r"[^\w\d]+", enriched, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]: break
    if not fts_q: return []
    fts_query = " OR ".join(f'"{t}"' for t in fts_q)
    rows = _fts.execute(
        "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? ORDER BY bm25(docs) LIMIT ?",
        (fts_query, k),
    ).fetchall()
    return [(did, -score) for did, score in rows]


# -----------------------------------------------------------------------------
# Phase 5.E — NEW multi-language BM25 search.
# -----------------------------------------------------------------------------
# For each language L, builds a language-specific query string from:
#   - the original English query (always present; English query tokens still
#     match against English concepts_en mixed into court search_text rows)
#   - language-appropriate enrichment terms from the LLM-produced `targets`
# Runs FTS5 on that index, takes its top-k_per_lang. Merges by `did` taking
# max-score across languages, then returns the global top k_total.
#
# Score normalisation:
#   FTS5 bm25() returns a NEGATIVE score (more negative = more relevant). We
#   flip sign first (bigger = more relevant). Then divide by
#   log(corpus_size_for_lang + math.e) so a tiny language (~94k IT rows) and
#   a huge one (~1.4M DE rows) produce comparable score magnitudes.
def _build_lang_query(english_query, targets, lang):
    """Build the FTS5 query string for a given language."""
    parts = [english_query or ""]
    targets = targets or {}
    concept_en = list(targets.get("concept_targets_en") or [])
    term_de = list(targets.get("term_targets_de") or [])
    term_fr = list(targets.get("term_targets_fr") or [])
    if lang == "de":
        parts.extend(term_de)
        parts.extend(concept_en)
    elif lang == "fr":
        parts.extend(term_fr)
        parts.extend(concept_en)
    elif lang == "it":
        # No it-specific targets in the qexp schema; fall back to de+fr+en.
        parts.extend(term_de)
        parts.extend(term_fr)
        parts.extend(concept_en)
    else:  # "en" and any unexpected language
        parts.extend(concept_en)
    return " ".join(p for p in parts if p)


def _split_budget(k_total, lang_doc_count):
    """Allocate per-language budget proportional to docs in that language,
    with a small floor so tiny languages still contribute. Returns dict."""
    total_docs = sum(max(1, lang_doc_count[L]) for L in _LANGS)
    floor = max(1, k_total // 16)  # at least ~6% of budget per language
    raw = {L: max(floor, int(round(k_total * lang_doc_count[L] / total_docs)))
           for L in _LANGS}
    # Trim if floors caused over-allocation; never below the floor though.
    over = sum(raw.values()) - k_total
    if over > 0:
        # subtract from largest first
        for L in sorted(_LANGS, key=lambda x: -raw[x]):
            take = min(over, raw[L] - floor)
            if take <= 0: continue
            raw[L] -= take; over -= take
            if over <= 0: break
    return raw


def bm25_search_multilang(query, targets, k_total):
    """Run BM25 per language, merge by did with max-score, return top k_total.

    Args:
        query    : original English query string.
        targets  : dict from Phase 7 (statute_targets, term_targets_de,
                   term_targets_fr, concept_targets_en, ...).
        k_total  : total budget (CONFIG['budget_bm25']).

    Returns:
        list of (did, score) tuples, sorted by score desc, length <= k_total.
    """
    if not _fts_by_lang:
        return []
    budgets = _split_budget(k_total, _lang_doc_count)
    # enhance() runs on the original English query only (the corpus-derived
    # codes are language-agnostic statute codes like "stpo", "bgg" — these are
    # appended to every language's query).
    boosted = _enhance_codes(query or "")
    boost_str = " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)

    merged = {}  # did -> best normalised score
    for L in _LANGS:
        if _lang_doc_count[L] == 0:
            continue
        per_lang_query = _build_lang_query(query, targets, L)
        if boost_str:
            per_lang_query = per_lang_query + " " + boost_str
        # Tokenise for FTS5: alpha/num tokens >= min_len; cap to budget.
        fts_q = []
        seen = set()
        for tok in re.split(r"[^\w\d]+", per_lang_query, flags=re.UNICODE):
            if len(tok) < CONFIG["bm25_min_token_len"]:
                continue
            tl = tok.lower()
            if tl in seen:
                continue
            seen.add(tl)
            fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]:
                break
        if not fts_q:
            continue
        fts_query = " OR ".join(f'"{t}"' for t in fts_q)
        # Cross-language comparability: divide raw score by log(N_lang + e).
        denom = math.log(_lang_doc_count[L] + math.e)
        try:
            rows = _fts_by_lang[L].execute(
                "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? "
                "ORDER BY bm25(docs) LIMIT ?",
                (fts_query, budgets[L]),
            ).fetchall()
        except sqlite3.OperationalError as _e:
            # malformed query (e.g. all stop-words) — skip this language.
            print(f"  bm25[{L}] skipped: {_e}")
            continue
        for did, raw_score in rows:
            # FTS5 bm25 is negative (lower = more relevant). Flip sign so
            # bigger = more relevant, then normalise by language size.
            norm = (-raw_score) / denom
            prev = merged.get(did)
            if prev is None or norm > prev:
                merged[did] = norm

    if not merged:
        return []
    out = sorted(merged.items(), key=lambda kv: -kv[1])[:k_total]
    return out


Building in-memory FTS5 (legacy, single index) over 2,649,348 docs...
  inserted 500,000 (11.5s)
  inserted 1,000,000 (27.2s)
  inserted 1,500,000 (44.2s)
  inserted 2,000,000 (61.5s)
  inserted 2,500,000 (77.7s)
FTS5 (legacy) built: 2,649,348 rows in 84.0s
Building per-language FTS5 indices (de/fr/it/en)...
Per-language FTS5 built in 70.6s
  fts[de]: 1,593,249 docs
  fts[fr]: 793,023 docs
  fts[it]: 125,583 docs
  fts[en]: 137,493 docs


# Phase 6 — Vector channel setup

## 6.1 What this cell does

Loads the 27 fp16 embedding chunks (~21 GB total) into a single `E_GPU`
tensor of shape (2.65 M, 4096). Provides `vector_search(q_emb, k)` doing
brute-force matmul on GPU.

## 6.2 Why brute-force GPU and not FAISS-IVF

- Blackwell has 95 GB VRAM; corpus E_GPU at fp16 fits in ~22 GB.
- Matmul `E_GPU @ q` is ~50 ms on GPU; comparable to FAISS-IVF query.
- No quantization recall loss, no index-build time.
- Per `personal_observations.md` Obs 3: dense embedding alone caps at
  R@1000 = 0.289 on val regardless of index — the bottleneck is "wrong kind
  of relationship for cosine similarity", not retrieval algorithm.

## 6.3 Manifest mapping

`qwen3_8b_unified_manifest.parquet` maps doc_id ↔ row_index in E_GPU.

**Expected:** 2.65 M rows in manifest, ~99.9% mapping coverage to our doc_ids.

**Failure modes:**
- `EMB_AVAILABLE = False` → vector channels skipped. Anchor + bedrock + BM25
  still run but recall drops.
- VRAM OOM during chunk concat → torch.cat allocates 2× peak; switch to
  in-place writes if chunk count grows.

In [12]:
VECTOR_OK = False
E_GPU = None
my_did_for_row = None
row_for_did = None

if EMB_AVAILABLE:
    import torch as _torch, numpy as _np, pandas as _pd
    print("Loading manifest...")
    _t = time.time()
    _man = _pd.read_parquet(PATHS["emb_manifest"])
    print(f"  manifest rows: {len(_man):,}, cols: {list(_man.columns)}")
    # Build row_index -> our doc_id mapping
    row_for_did = {}; my_did_for_row = [None] * len(_man)
    for _, r in _man.iterrows():
        cit = r.get("citation") or ""
        fam = r.get("family") or ""
        ridx = int(r.get("row_index", -1))
        if ridx < 0: continue
        # Resolve to a doc_id in our cit_to_doc_ids based on family
        candidates = cit_to_doc_ids.get(cit, [])
        for d in candidates:
            if doc_meta.get(d, {}).get("family") == fam:
                row_for_did[d] = ridx
                if ridx < len(my_did_for_row):
                    my_did_for_row[ridx] = d
                break
    n_mapped = sum(1 for x in my_did_for_row if x is not None)
    print(f"  manifest->my_did mapping: {n_mapped:,}/{len(_man):,} ({100*n_mapped/len(_man):.1f}%) in {time.time()-_t:.1f}s")

    # Concat chunks to GPU
    print(f"  loading chunks to GPU (~21 GB)...")
    _t = time.time()
    _chunks = sorted(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
    arrs = []
    for cp in _chunks:
        arrs.append(_torch.from_numpy(_np.load(cp)).to("cuda", non_blocking=True))
    E_GPU = _torch.cat(arrs, dim=0); del arrs
    free_vram = (_torch.cuda.get_device_properties(0).total_memory
                 - _torch.cuda.memory_allocated()) / 1024**3
    print(f"  E_GPU shape={tuple(E_GPU.shape)} dtype={E_GPU.dtype}, "
          f"VRAM used={_torch.cuda.memory_allocated()/1024**3:.1f} GB, "
          f"free={free_vram:.1f} GB, "
          f"load {time.time()-_t:.1f}s")

    def vector_search(q_emb, k):
        if E_GPU is None: return []
        with _torch.no_grad():
            q = q_emb.to(E_GPU.device, dtype=E_GPU.dtype)
            q = q / (q.norm(dim=-1, keepdim=True) + 1e-9)
            scores = E_GPU @ q
            top_v, top_i = _torch.topk(scores, k=min(k, scores.shape[0]))
        out = []
        for s, i in zip(top_v.cpu().tolist(), top_i.cpu().tolist()):
            d = my_did_for_row[i]
            if d is not None: out.append((d, float(s)))
        return out

    VECTOR_OK = True
else:
    print("[skip] EMB_AVAILABLE = False; vector channels will be skipped.")
    def vector_search(q_emb, k): return []

Loading manifest...
  manifest rows: 2,652,248, cols: ['doc_id', 'family', 'citation', 'row_index']
  manifest->my_did mapping: 2,649,348/2,652,248 (99.9%) in 39.3s
  loading chunks to GPU (~21 GB)...
  E_GPU shape=(2652248, 4096) dtype=torch.float16, VRAM used=20.2 GB, free=74.7 GB, load 481.5s


## 6.4 Embedding model for query

**What:** Load Qwen3-Embedding-8B via SentenceTransformer for **query-side**
encoding (corpus-side is pre-encoded).

**Why:** We need the same model that produced the corpus embeddings, with
its canonical instruction prefix.

**VRAM:** ~16 GB. Loaded after Qwen3-32B is freed (Phase 7).

**Failure modes:**
- HF download blocked → preflight `huggingface_hub.snapshot_download` once.
- Model weights mismatch (8B vs 8B-Embedding) → ensure model id is exactly
  `Qwen/Qwen3-Embedding-8B`.

In [13]:
EMB_MODEL = None
def encode_query(text):
    return EMB_MODEL.encode(
        [text],
        prompt_name="query",
        convert_to_tensor=True,
        normalize_embeddings=True,
    )[0]

# Phase 7 — Query expansion (Qwen3-32B)

Load Qwen3-32B **once**, loop over every query in `ALL_QUERIES`, free the model. Produces `ALL_TARGETS[query_id]` (LLM-named statutes, concepts, terms, legal-area keywords).


In [14]:
QEXP_PROMPT_SYSTEM = (
    "You are a Swiss legal-research assistant. You enumerate broadly across "
    "the most relevant Swiss federal codes (StPO, StGB, BGG, ZGB, OR, ZPO, BV, "
    "EMRK, IPRG, IRSG, AHVG, IVG, AsylG, AIG, DBG, StHG, KVG, UVG, AVIG, PatG, "
    "MSchG, URG, FINIG, FINMAG, BankG, KKG, SchKG, VwVG, FZG, BVG, BPV, BetmG, "
    "RPG, NHG, USG, GSchG, SVG, GwG, etc). You output ONLY a single JSON "
    "object — no prose, no markdown, no code fences."
)

QEXP_PROMPT_USER = '''Schema (exact keys, all required):

{
  "statute_targets":      [10-20 "Art. N CODE" strings, broad coverage of relevant codes],
  "case_targets":         [list of "BGE V D P" or "1B_N/Y" dockets — empty if uncertain],
  "concept_targets_en":   [15-25 English legal concepts (technical AND procedural)],
  "term_targets_de":      [15-25 German legal terms in original spelling],
  "term_targets_fr":      [10-20 French legal terms],
  "legal_area_keywords":  [3-6 short legal-area phrases]
}

Hard rules:
- Output ONLY the JSON object. No surrounding text, no markdown, no code fences.
- Each list MUST have at least the minimum number of items.
- Do NOT invent specific BGE volumes or dockets you are uncertain about.

Example A for "Does an unemployed worker keep insurance benefits when he refuses a job offer that is below his prior wage?":
{"statute_targets":["Art. 16 AVIG","Art. 17 AVIG","Art. 30 AVIG","Art. 30 Abs. 1 AVIG","Art. 16 Abs. 2 AVIG","Art. 22 AVIG","Art. 23 AVIG","Art. 8 AVIG","Art. 95 AVIG","Art. 11 AVIG"],
"case_targets":[],
"concept_targets_en":["unemployment insurance","suitable employment","wage protection","willingness to work","refusal of suitable work","reduction of benefits","good cause","unemployment compensation","insured earnings","obligation to accept","sanction","placement","right to compensation","intermediate earnings","admissibility threshold"],
"term_targets_de":["Arbeitslosenversicherung","zumutbare Arbeit","Lohnvergleich","Vermittlungsfähigkeit","Ablehnung","Einstellung in der Anspruchsberechtigung","Versicherungsleistungen","Arbeitslosenentschädigung","Zwischenverdienst","Vermittlungsbemühungen","Arbeitsbemühungen","versicherter Verdienst","Selbstverschulden","Sanktion","Verfügung"],
"term_targets_fr":["assurance-chômage","emploi convenable","comparaison de salaire","aptitude au placement","refus","suspension du droit à l'indemnité","indemnisation","gain intermédiaire","obligation","sanction"],
"legal_area_keywords":["unemployment insurance","social insurance","labour market","social security law"]}

Example B for "Is a will written on lined notebook paper and signed only on the last page valid under Swiss inheritance law?":
{"statute_targets":["Art. 505 ZGB","Art. 498 ZGB","Art. 499 ZGB","Art. 519 ZGB","Art. 520 ZGB","Art. 6 ZGB","Art. 467 ZGB","Art. 468 ZGB","Art. 522 ZGB","Art. 540 ZGB"],
"case_targets":[],
"concept_targets_en":["holographic will","testator","handwriting","signature","testamentary capacity","formal validity","formal requirements","invalidity","challenge of will","disposition mortis causa","heirship","forced heirship","compulsory portion","inheritance","reduction action"],
"term_targets_de":["eigenhändige Verfügung","Testament","Erblasser","Handschrift","Unterschrift","Verfügungsfähigkeit","Formvorschriften","Ungültigerklärung","Anfechtung","letztwillige Verfügung","Erbe","Pflichtteil","Erbschaft","Herabsetzung","Verfügung von Todes wegen"],
"term_targets_fr":["testament olographe","testateur","écriture","signature","capacité de disposer","conditions de forme","nullité","action en réduction","disposition pour cause de mort","héritage","réserve héréditaire"],
"legal_area_keywords":["inheritance law","succession","testamentary law","civil law"]}

Now produce the JSON for this query (JSON object only, no other text):
{QUERY}
'''

import time, gc
import re as _re
import json as _json
import torch as _torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def parse_targets_json(text):
    """Robust extractor: markdown-fence -> brace-balanced scan -> greedy."""
    if not text:
        return None
    # 1) ```json {...} ``` or ``` {...} ```
    fence = _re.search(r"```(?:json|JSON)?\s*(\{.*?\})\s*```", text, _re.S)
    if fence:
        try:
            return _json.loads(fence.group(1))
        except Exception:
            pass
    # 2) brace-balanced scan from each '{'
    n = len(text); i = 0
    while i < n:
        i = text.find("{", i)
        if i < 0:
            break
        depth = 0; in_str = False; esc = False; j = i
        while j < n:
            c = text[j]
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                in_str = not in_str
            elif not in_str:
                if c == "{":
                    depth += 1
                elif c == "}":
                    depth -= 1
                    if depth == 0:
                        cand = text[i:j+1]
                        try:
                            return _json.loads(cand)
                        except Exception:
                            break
            j += 1
        i += 1
    # 3) greedy fallback
    m = _re.search(r"\{.*\}", text, _re.S)
    if m:
        try:
            return _json.loads(m.group(0))
        except Exception:
            pass
    return None


def normalize_targets(d):
    """Coerce whatever the model returned into the expected schema."""
    keys = ("statute_targets", "case_targets", "concept_targets_en",
            "term_targets_de", "term_targets_fr", "legal_area_keywords")
    out = {k: [] for k in keys}
    if not isinstance(d, dict):
        return out
    for k in keys:
        v = d.get(k)
        if v is None:
            continue
        if isinstance(v, str):
            out[k] = [v.strip()] if v.strip() else []
        elif isinstance(v, list):
            flat = []
            for item in v:
                if isinstance(item, str) and item.strip():
                    flat.append(item.strip())
                elif isinstance(item, dict):
                    for vv in item.values():
                        if isinstance(vv, str) and vv.strip():
                            flat.append(vv.strip()); break
            out[k] = flat
    return out


_qmodel = CONFIG["qwen_query_model"]
print(f"[qexp] loading {_qmodel} (~65 GB bf16)...")
_t = time.time()
qtok = AutoTokenizer.from_pretrained(_qmodel)
qmod = AutoModelForCausalLM.from_pretrained(
    _qmodel,
    dtype=_torch.bfloat16,
    device_map="auto",
)
qmod.eval()
print(f"[qexp] model loaded in {time.time() - _t:.1f}s")

_input_device = next(p.device for p in qmod.parameters() if p.device.type != "meta")

ALL_TARGETS = {}
ALL_RAW_RESPONSES = {}

for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    _t0 = time.time()

    _messages = [
        {"role": "system", "content": QEXP_PROMPT_SYSTEM},
        {"role": "user",   "content": QEXP_PROMPT_USER.replace("{QUERY}", qtext)},
    ]
    _inp = qtok.apply_chat_template(
        _messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )
    _inp = {k: v.to(_input_device) for k, v in _inp.items()}
    _prompt_len = _inp["input_ids"].shape[1]

    # Pass 1: deterministic.
    with _torch.no_grad():
        _out = qmod.generate(
            **_inp,
            max_new_tokens=CONFIG["qwen_max_new_tokens"],
            do_sample=False,
            pad_token_id=qtok.eos_token_id,
        )
    _resp = qtok.decode(_out[0][_prompt_len:], skip_special_tokens=True)
    raw = parse_targets_json(_resp)
    targets = normalize_targets(raw)
    n_items = sum(len(v) for v in targets.values())

    # Pass 2: sampling retry on parse failure OR weak result.
    retried = False
    if raw is None or n_items < 10:
        retried = True
        with _torch.no_grad():
            _out2 = qmod.generate(
                **_inp,
                max_new_tokens=int(CONFIG["qwen_max_new_tokens"] * 1.5),
                do_sample=True,
                temperature=0.4,
                top_p=0.95,
                pad_token_id=qtok.eos_token_id,
            )
        _resp2 = qtok.decode(_out2[0][_prompt_len:], skip_special_tokens=True)
        raw2 = parse_targets_json(_resp2)
        if raw2 is not None:
            targets2 = normalize_targets(raw2)
            n_items2 = sum(len(v) for v in targets2.values())
            if n_items2 > n_items:
                targets, _resp, n_items = targets2, _resp2, n_items2

    ALL_RAW_RESPONSES[qid] = _resp
    ALL_TARGETS[qid] = targets

    flag = "[retry]" if retried else "       "
    n_stat = len(targets["statute_targets"])
    n_ce   = len(targets["concept_targets_en"])
    n_td   = len(targets["term_targets_de"])
    n_tf   = len(targets["term_targets_fr"])
    n_la   = len(targets["legal_area_keywords"])
    print(f"[qexp] {qid}  {time.time()-_t0:5.1f}s {flag}  "
          f"stat={n_stat:>2} ce={n_ce:>2} td={n_td:>2} tf={n_tf:>2} la={n_la:>2}")

    if n_items == 0:
        # Print enough of the raw response to diagnose the failure.
        print(f"  [qexp][WARN] {qid}: empty targets even after retry. "
              f"Raw response (first 800 chars):")
        print("  " + repr(_resp[:800]))

# ---------------------------------------------------------------------------
# HyDE pass — hypothetical legal-prose answer per query.
# Generated by the same Qwen3-32B (model still loaded). The answer paragraph
# matches the *shape* of corpus paragraphs (long-form legalese in DE/FR/IT
# context, statute references inline), making the embedding closer to gold
# paragraphs than the raw English question vector.
# ---------------------------------------------------------------------------

QHYDE_PROMPT_SYSTEM = (
    "You are a Swiss legal expert. Given an English legal question, you write "
    "a single 4-6 sentence answer paragraph in the style of a Bundesgericht / "
    "Tribunal fédéral consideration. Cite the most relevant Swiss articles "
    "inline as 'Art. N CODE'. Use precise legal terminology and inline German "
    "AND French legal terms in parentheses where natural (e.g. 'pre-trial "
    "detention (Untersuchungshaft / détention provisoire)'). Output ONLY the "
    "answer paragraph — no preamble, no headings, no markdown."
)

QHYDE_PROMPT_USER = (
    "Write a 4-6 sentence hypothetical answer paragraph in Swiss-Federal-"
    "Tribunal style. Cite the most relevant Swiss articles inline. Include "
    "German and French legal terms in parentheses where natural.\n\n"
    "Query: {QUERY}\n\nAnswer paragraph:"
)

ALL_HYDE = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    _th = time.time()
    _msgs_h = [
        {"role": "system", "content": QHYDE_PROMPT_SYSTEM},
        {"role": "user",   "content": QHYDE_PROMPT_USER.replace("{QUERY}", qtext)},
    ]
    _inp_h = qtok.apply_chat_template(
        _msgs_h,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )
    _inp_h = {k: v.to(_input_device) for k, v in _inp_h.items()}
    _plen_h = _inp_h["input_ids"].shape[1]
    with _torch.no_grad():
        _out_h = qmod.generate(
            **_inp_h,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=qtok.eos_token_id,
        )
    _hyde = qtok.decode(_out_h[0][_plen_h:], skip_special_tokens=True).strip()
    # Strip an obvious leading "Answer:" / "Antwort:" if the model emits one.
    for _pfx in ("Answer paragraph:", "Answer:", "A:"):
        if _hyde.startswith(_pfx):
            _hyde = _hyde[len(_pfx):].strip()
    # If the model produced nothing useful, fall back to the raw query so the
    # downstream encoder always has text to embed.
    if len(_hyde) < 40:
        print(f"[hyde] {qid}  {time.time()-_th:5.1f}s  [WARN short] len={len(_hyde)}; falling back to QUERY text")
        _hyde = qtext
    else:
        print(f"[hyde] {qid}  {time.time()-_th:5.1f}s  {len(_hyde):>4} chars  '{_hyde[:120]}...'")
    ALL_HYDE[qid] = _hyde

# Free Qwen3-32B from VRAM.
del qmod, qtok, _inp, _out, _inp_h, _out_h
gc.collect()
_torch.cuda.empty_cache()
if _torch.cuda.is_available():
    _torch.cuda.synchronize()
    print(f"\n[qexp] freed Qwen3-32B; CUDA mem={_torch.cuda.memory_allocated() / 1024**3:.1f} GB")
else:
    print("\n[qexp] freed Qwen3-32B")


[qexp] loading Qwen/Qwen3-32B (~65 GB bf16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[qexp] model loaded in 205.4s
[qexp] val_001   56.5s          stat=20 ce=20 td=20 tf=20 la= 6
[qexp] val_002   52.8s          stat=20 ce=24 td=24 tf=24 la= 6
[qexp] val_003   51.9s          stat=20 ce=25 td=25 tf=23 la= 6
[qexp] val_004   44.2s          stat=20 ce=20 td=20 tf=19 la= 6
[qexp] val_005  108.7s [retry]  stat=20 ce=25 td=25 tf=22 la= 6
[qexp] val_006   51.9s          stat=20 ce=24 td=24 tf=21 la= 6
[qexp] val_007   50.9s          stat=20 ce=25 td=25 tf=24 la= 6
[qexp] val_008   54.9s          stat=20 ce=25 td=26 tf=21 la= 6
[qexp] val_009   46.1s          stat=20 ce=25 td=25 tf=21 la= 5
[qexp] val_010   50.3s          stat=20 ce=20 td=20 tf=20 la= 6
[hyde] val_001   15.3s  1381 chars  'The extension of pre-trial detention (Untersuchungshaft / détention provisoire) under Art. 221 Abs. 1 lit. b StPO must b...'
[hyde] val_002   15.9s  1468 chars  'The claimant’s entitlement to vocational rehabilitation and invalidity insurance benefits under Art. 17 LAI (BVG) must b...'
[hyde]

## 7.6 Encode queries (Qwen3-Embedding-8B)

Load Qwen3-Embedding-8B **once**, loop over every query. Produces `ALL_Q_EMB_RAW[qid]` and `ALL_Q_EMB_ENRICHED[qid]`.


In [15]:
print("Loading Qwen3-Embedding-8B...")
from sentence_transformers import SentenceTransformer
EMB_MODEL = SentenceTransformer(CONFIG["vector_emb_model"])
print(f"  done")

ALL_Q_EMB_RAW = {}
ALL_Q_EMB_ENRICHED = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    tgt = ALL_TARGETS.get(qid, {})
    enriched_bits = (
        (tgt.get("term_targets_de") or [])
      + (tgt.get("term_targets_fr") or [])
      + (tgt.get("concept_targets_en") or [])
    )
    enriched_query = qtext + " " + " ".join(enriched_bits[:60])
    ALL_Q_EMB_RAW[qid] = encode_query(qtext)
    ALL_Q_EMB_ENRICHED[qid] = encode_query(enriched_query)
    print(f"  {qid:<8}  raw + enriched   ({len(qtext)} -> {len(enriched_query)} chars, "
          f"+{min(60, len(enriched_bits))} kw)")

# Encode HyDE answer paragraphs as a third vector query.
ALL_Q_EMB_HYDE = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    hyde_text = ALL_HYDE.get(qid) or q["query_text"]
    ALL_Q_EMB_HYDE[qid] = encode_query(hyde_text)
    print(f"  {qid:<8}  hyde ({len(hyde_text)} chars)")


Loading Qwen3-Embedding-8B...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

  done
  val_001   raw + enriched   (1068 -> 2309 chars, +60 kw)
  val_002   raw + enriched   (1636 -> 3014 chars, +60 kw)
  val_003   raw + enriched   (1677 -> 2740 chars, +60 kw)
  val_004   raw + enriched   (1147 -> 2159 chars, +59 kw)
  val_005   raw + enriched   (1581 -> 2837 chars, +60 kw)
  val_006   raw + enriched   (1437 -> 2774 chars, +60 kw)
  val_007   raw + enriched   (1742 -> 2791 chars, +60 kw)
  val_008   raw + enriched   (1527 -> 2622 chars, +60 kw)
  val_009   raw + enriched   (1116 -> 2227 chars, +60 kw)
  val_010   raw + enriched   (1695 -> 3015 chars, +60 kw)
  val_001   hyde (1381 chars)
  val_002   hyde (1468 chars)
  val_003   hyde (1234 chars)
  val_004   hyde (1045 chars)
  val_005   hyde (1210 chars)
  val_006   hyde (1133 chars)
  val_007   hyde (1311 chars)
  val_008   hyde (1341 chars)
  val_009   hyde (1100 chars)
  val_010   hyde (1301 chars)


# Phase 8 — Channels

Define the channel-runner function. Every query goes through this identical pipeline — no per-id specialisation.


In [16]:
from collections import Counter

def channel_law_direct(canon_set, idx, budget):
    counter = Counter()
    for canon in canon_set:
        for did in idx[canon]: counter[did] += 1
    items = counter.most_common()
    return items if budget is None else items[:budget]

def channel_court_statute(statute_canons, idx, idx_count=None, doc_meta=None, budget=None):
    """Score court rows for an LLM-named statute canonical set.

    Score(did) = (sum over matched canons of 1/log(2 + global_count[canon]))
                 * (1 + 0.3 * (n_matches - 1))         # multi-match bonus
                 * paragraph_role_weight(did)          # 0.4 / 0.6 / 1.0 / 1.5

    Backwards-compatible: idx_count and doc_meta are optional. If idx_count is
    None (e.g. cell 12 hasn't been re-run with the new patch), counts are derived
    from idx on the fly. If doc_meta is None, role weighting is skipped (1.0).
    """
    import math as _math
    if idx_count is None:
        idx_count = {c: len(idx[c]) for c in statute_canons if c in idx}

    # paragraph_role -> multiplicative weight
    _ROLE_W = {
        "legal_standard": 1.5, "reasoning": 1.5,
        "application":    1.5, "holding":   1.5,
        "facts":              1.0, "procedural_history": 1.0,
        "citation":           1.0, "neutral_default":    1.0,
        "costs":        0.6, "disposition": 0.6, "notification": 0.6,
        "neutral":      0.4,
    }

    # Per-doc accumulator: {did: [matches_so_far, summed_specificity_weight]}
    per_doc = {}
    for canon in statute_canons:
        dids = idx.get(canon)
        if not dids:
            continue
        cnt = idx_count.get(canon)
        if cnt is None:
            cnt = len(dids)
        w_canon = 1.0 / _math.log(2 + cnt)
        for did in dids:
            slot = per_doc.get(did)
            if slot is None:
                per_doc[did] = [1, w_canon]
            else:
                slot[0] += 1
                slot[1] += w_canon

    if not per_doc:
        return []

    scored = []
    for did, (n_matches, base_w) in per_doc.items():
        score = base_w * (1.0 + 0.3 * (n_matches - 1))
        if doc_meta is not None:
            meta = doc_meta.get(did) or {}
            role = meta.get("paragraph_role")
            if role is None or role == "":
                rw = 0.4
            else:
                rw = _ROLE_W.get(role, 1.0)
            score *= rw
        scored.append((did, float(score)))

    # Stable deterministic ordering: descending score, ascending doc_id on ties.
    scored.sort(key=lambda x: (-x[1], x[0]))
    if budget is None:
        return scored
    return scored[:budget]

import math as _math_v74

_SUBSTANTIVE_ROLES = {"reasoning", "legal_standard", "application", "holding"}

def _judgment_factor(cb, idx_judgment_importance):
    """sqrt(1 + log(1 + importance)). Default 1.0 for unknown / 0 importance."""
    imp = idx_judgment_importance.get(cb, 0) if cb else 0
    if imp <= 0:
        return 1.0
    return _math_v74.sqrt(1.0 + _math_v74.log(1.0 + float(imp)))

def _role_boost(doc_meta, did):
    m = doc_meta.get(did) or {}
    role = (m.get("paragraph_role") or "").strip().lower()
    return 1.5 if role in _SUBSTANTIVE_ROLES else 1.0

def channel_sibling(seed_doc_ids, idx_court_base, idx_judgment_importance,
                    doc_meta, budget):
    # 1) seed_count[cb] = how many caught seeds share court_base cb
    seed_count = Counter()
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb = m.get("court_base")
        if cb: seed_count[cb] += 1
    # 2) emit every sibling row with score = seed_count * judgment_factor * role_boost
    seed_set = set(seed_doc_ids)
    scored = {}
    for cb, cnt in seed_count.items():
        factor = _judgment_factor(cb, idx_judgment_importance)
        for s in idx_court_base.get(cb, ()):
            if s in seed_set: continue
            sc = float(cnt) * factor * _role_boost(doc_meta, s)
            # keep best score per doc (a row only belongs to one cb)
            if sc > scored.get(s, 0.0):
                scored[s] = sc
    # 3) deterministic order: score desc, doc_id asc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_forward(seed_doc_ids, idx_graph_out, idx_judgment_importance,
                          doc_meta, budget):
    # edge-count per target (how many distinct seeds point to it)
    seed_set = set(seed_doc_ids)
    edge_count = Counter()
    for did in seed_doc_ids:
        for t in idx_graph_out.get(did, ()):
            edge_count[t] += 1
    for d in seed_set:
        edge_count.pop(d, None)
    # weight each target by importance of its OWN judgment + role boost
    scored = {}
    for t, cnt in edge_count.items():
        m = doc_meta.get(t) or {}
        cb_of_t = m.get("court_base")
        factor = _judgment_factor(cb_of_t, idx_judgment_importance)
        sc = float(cnt) * factor * _role_boost(doc_meta, t)
        scored[t] = sc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_reverse(seed_doc_ids, idx_graph_in, idx_judgment_importance,
                          doc_meta, budget,
                          landmark_imp_threshold=5):
    """Reverse-graph expansion gated to landmark seeds.
    Only follows incoming-edges from seeds whose judgment has
    importance >= landmark_imp_threshold (default 5). Each contributing
    edge is weighted by the SEED-judgment importance, so landmark seeds
    dominate.
    """
    seed_set = set(seed_doc_ids)
    scored = {}
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb_seed = m.get("court_base")
        imp_seed = idx_judgment_importance.get(cb_seed, 0) if cb_seed else 0
        if imp_seed < landmark_imp_threshold:
            continue
        factor = _judgment_factor(cb_seed, idx_judgment_importance)
        for s in idx_graph_in.get(did, ()):
            if s in seed_set: continue
            inc = factor * _role_boost(doc_meta, s)
            scored[s] = scored.get(s, 0.0) + inc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_2hop(seed_doc_ids, idx_graph_out, budget):
    seed_set = set(seed_doc_ids)
    intermediate = set()
    for did in seed_doc_ids:
        intermediate.update(idx_graph_out.get(did, ()))
    intermediate -= seed_set
    counter = Counter()
    for x in intermediate:
        for t in idx_graph_out.get(x, ()):
            if t in seed_set: continue
            counter[t] += 1
    for d in intermediate:
        counter.pop(d, None)
    return counter.most_common(budget)

_CONCEPT_STOPWORDS_EN = frozenset({
    "a","an","the","of","in","on","at","by","for","with","to","from",
    "and","or","but","is","are","was","were","be","been","being",
    "has","have","had","do","does","did","no","not",
})
_CONCEPT_TOKEN_SPLIT_RE = re.compile(r"[^\w]+", re.UNICODE)

def _concept_tokens(s):
    """Whitespace+punct split, lowercase. Returns full token list."""
    if not s: return []
    return [t for t in _CONCEPT_TOKEN_SPLIT_RE.split(s.lower()) if t]

def _concept_meaningful_tokens(s):
    return [t for t in _concept_tokens(s) if t not in _CONCEPT_STOPWORDS_EN]

def expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25):
    """Replace strict-substring matcher with a weighted scorer.

    Combines three signals against each corpus concept:
      1. Exact match               -> weight 1.0
      2. Substring (either dir)    -> weight 0.85 * shorter/longer
         (must also share >=1 meaningful token; otherwise a stopword
         only embedding like 'of detention' could hijack 'of').
      3. Token overlap             -> weight shared/max(q_tok, c_tok)
         after stripping English stopwords from BOTH sides.

    Returns list[(corpus_concept, weight)] sorted by weight desc,
    capped at top_k per query concept (max weight kept across
    multiple query concepts hitting the same corpus concept).
    """
    W_EXACT = 1.0
    W_SUBSTR_MAX = 0.85  # capped below exact so true matches dominate

    keys = list(corpus_concept_keys)
    # Pre-tokenize the corpus once: corpus_concept -> (set_of_tokens, n_tokens)
    corpus_meaningful = {}
    for k in keys:
        m = _concept_meaningful_tokens(k)
        if m:
            corpus_meaningful[k] = (set(m), len(m))

    best_weight = {}  # corpus_concept -> max weight across query concepts

    for raw in llm_concepts or []:
        c = (raw or "").lower().strip()
        if not c or len(c) < 4:
            continue
        c_meaningful = _concept_meaningful_tokens(c)
        if not c_meaningful:
            # Pure-stopword query (e.g. "of the") yields no matches.
            continue
        c_set = set(c_meaningful)
        c_len = len(c_meaningful)
        c_chars = len(c)

        per_query = []  # (weight, length_diff, corpus_concept)

        # 1. Exact match
        if c in corpus_concept_keys:
            per_query.append((W_EXACT, 0, c))

        # 2. Substring match (either direction), gated on shared
        #    meaningful token to block stopword-only bridges.
        for cv in keys:
            if cv == c:
                continue
            if c in cv or cv in c:
                cv_info = corpus_meaningful.get(cv)
                if not cv_info:
                    continue
                cv_set, cv_len = cv_info
                if not (c_set & cv_set):
                    # only stopword/character overlap; reject
                    continue
                shorter = min(c_chars, len(cv))
                longer  = max(c_chars, len(cv))
                if longer <= 0:
                    continue
                w = W_SUBSTR_MAX * (shorter / longer)
                per_query.append((w, abs(len(cv) - c_chars), cv))

        # 3. Token overlap (stopwords already stripped on both sides)
        for cv, (cv_set, cv_len) in corpus_meaningful.items():
            if cv == c:
                continue
            shared = c_set & cv_set
            if not shared:
                continue
            denom = max(c_len, cv_len)
            if denom <= 0:
                continue
            w = len(shared) / denom
            per_query.append((w, abs(cv_len - c_len), cv))

        # Dedupe within this query concept: max weight per corpus key.
        local_best = {}
        for w, ld, cv in per_query:
            cur = local_best.get(cv)
            if cur is None or w > cur[0] or (w == cur[0] and ld < cur[1]):
                local_best[cv] = (w, ld)

        # Rank: weight desc, then length-diff asc, then alphabetical.
        ranked = sorted(local_best.items(),
                        key=lambda kv: (-kv[1][0], kv[1][1], kv[0]))
        for cv, (w, _) in ranked[:top_k]:
            prior = best_weight.get(cv, 0.0)
            if w > prior:
                best_weight[cv] = w

    # Final ordering: weight desc, then alphabetical for determinism.
    return sorted(best_weight.items(), key=lambda kv: (-kv[1], kv[0]))

def channel_concept(expanded_weighted, idx, budget):
    """Consume list[(concept, weight)] from expand_concepts_weighted.

    Each doc accumulates the sum of weights from every matched
    corpus concept it carries. Top `budget` by accumulated score.
    Backward-compatible: tolerates a list of plain strings (legacy)
    by treating each as weight=1.0.
    """
    scores = {}
    for item in expanded_weighted or []:
        if isinstance(item, tuple):
            tok, w = item
        else:
            tok, w = item, 1.0
        if not tok:
            continue
        for did in idx.get(tok, ()):
            scores[did] = scores.get(did, 0.0) + float(w)
    if not scores:
        return []
    # Sort by score desc, then doc_id for determinism. Cap at budget.
    items = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    if budget is not None:
        items = items[:budget]
    return items

def channel_term(targets, idx, idx_lemma, budget,
                 corpus_keys=None, lemma_score=0.7,
                 min_substring_len=4):
    """Term_orig channel — exact + lemma + substring fused scorer.

    Score model (per query-term/corpus-term pair, MAX over paths):
      exact         -> 1.0
      lemma equal   -> `lemma_score` (default 0.7) (DE-only; we still try FR/IT
                       through term_lemma but lemma_score is conservative)
      Q in T        -> len(Q) / len(T)
      T in Q        -> len(T) / len(Q)   (only if len(T) >= min_substring_len)
    Per doc_id we sum scores across all matching pairs and return the
    top-`budget` doc_ids by sum.
    """
    if corpus_keys is None:
        # Fallback: derive surface-form keys from the exact-match index. Slower
        # to construct on the fly but keeps callers without the precomputed set
        # working.
        corpus_keys = list(idx.keys())
    else:
        corpus_keys = list(corpus_keys)

    # Collect normalized query terms (DE first, then FR; we treat both
    # symmetrically — substring matching is language-agnostic, lemma logic
    # is most reliable for DE but doesn't actively hurt FR/IT because the
    # lemma function is identity for tokens with no removable suffix).
    q_terms = []
    for key in ("term_targets_de", "term_targets_fr"):
        for raw in targets.get(key, []) or []:
            tok = norm_token(raw, CONFIG["lowercase_terms"])
            if tok:
                q_terms.append(tok)
    if not q_terms:
        return []

    # doc_id -> accumulated score (max-per-pair, summed across query terms)
    score = Counter()

    for Q in q_terms:
        Q_lemma = term_lemma(Q)
        Q_len = len(Q)
        # Per-doc max for this single query term — prevents two paths
        # (e.g. exact + lemma) from double-counting the same doc.
        per_q = {}
        def _bump(did, s):
            if s > per_q.get(did, 0.0):
                per_q[did] = s

        # 1. Exact match — strongest signal, score 1.0.
        if Q in idx:
            for did in idx[Q]:
                _bump(did, 1.0)

        # 2. Lemma match — only fire when lemma differs from surface or when
        #    Q's lemma keys are present. Score 0.7 (capped below exact).
        if Q_lemma and Q_lemma in idx_lemma:
            for did in idx_lemma[Q_lemma]:
                _bump(did, lemma_score)

        # 3. Substring scan — for each corpus term T, compute the longer
        #    of Q⊂T and T⊂Q. Skip the exact-equal case (already scored).
        if Q_len >= 1:
            for T in corpus_keys:
                if T == Q:
                    continue
                T_len = len(T)
                s = 0.0
                if Q in T:
                    s = Q_len / T_len      # len(Q) / len(T)
                elif T_len >= min_substring_len and T in Q:
                    s = T_len / Q_len      # len(T) / len(Q)
                if s <= 0.0:
                    continue
                # Each corpus term T may map to many doc_ids; bump them all.
                for did in idx.get(T, ()):  # exact-form posting is canonical
                    _bump(did, s)

        # Roll the per-Q max scores into the cross-Q sum.
        for did, s in per_q.items():
            score[did] += s

    # most_common-style ordering by score, with deterministic tie-break on doc_id.
    items = sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_per_area_bedrock(legal_area_keywords, statute_target_codes,
                             per_area_canon_count, idx_law_direct, budget):
    if not legal_area_keywords: return []
    keys = [k.lower() for k in legal_area_keywords]
    selected_areas = set()
    for area in per_area_canon_count.keys():
        for k in keys:
            if k in area: selected_areas.add(area); break
    if not selected_areas: return []
    canon_score = Counter()
    for area in selected_areas:
        for canon, n in per_area_canon_count[area].most_common(CONFIG["per_area_top_n"]):
            canon_score[canon] = max(canon_score[canon], n)
    out = []; seen = set()
    for canon, _ in canon_score.most_common():
        canon_code = canon.split()[1] if canon and " " in canon else None
        if statute_target_codes and canon_code not in statute_target_codes:
            continue
        for did in idx_law_direct.get(canon, set()):
            if did not in seen:
                out.append((did, canon_score[canon])); seen.add(did)
        if len(out) >= budget: break
    return out[:budget]

def channel_statute_backprop(seed_court_dids, doc_statute_anchors, idx_law_direct, budget):
    import math as _math
    canon_to_courts = defaultdict(set)
    for did in seed_court_dids:
        for canon in doc_statute_anchors.get(did, set()):
            canon_to_courts[canon].add(did)
    # Specificity weighting: rare canons (small global court count) score more per
    # caught row than common procedural canons (Art. 100 BGG, Art. 9 BV, ...).
    # Idempotent against court_statute fix agent which may also define this:
    _idx_canon_ct = (locals().get('idx_court_statute_count')
                     or globals().get('idx_court_statute_count')
                     or {c: len(s) for c, s in idx_court_statute.items()})
    counter = Counter()
    for canon, court_set in canon_to_courts.items():
        n_caught = len(court_set)
        global_ct = _idx_canon_ct.get(canon, n_caught)
        score = n_caught * (1.0 / _math.log(2 + global_ct))
        for law_did in idx_law_direct.get(canon, set()):
            if counter[law_did] < score: counter[law_did] = score
    return counter.most_common(budget)

def channel_co_citation(targets, co_neighbours, idx_law_direct, idx_court_statute, budget):
    import math as _math
    # Use canon_count from cell 20 if available; otherwise derive from idx_court_statute.
    _canon_ct = (locals().get('canon_count')
                 or globals().get('canon_count')
                 or {c: len(s) for c, s in idx_court_statute.items()})
    counter = Counter()
    for raw in targets.get("statute_targets", []) or []:
        canon = statute_anchor_canonical(raw)
        if not canon: continue
        for nb, n in co_neighbours.get(canon, []):
            spec = 1.0 / _math.log(2 + _canon_ct.get(nb, 1))
            score = n * spec
            for did in idx_law_direct.get(nb, set()):
                if counter[did] < score: counter[did] = score
            for did in idx_court_statute.get(nb, set()):
                if counter[did] < score: counter[did] = score
    return counter.most_common(budget)

def channel_vector(q_emb, k):
    return vector_search(q_emb, k)

## 8.4 Channel runner (function)

`run_channels(QUERY_TEXT, targets, q_emb_raw, q_emb_enriched, gold_doc_set, total_gold)` — produces 14 channels for one query.


In [17]:
# Channel-runner — exact same logic as the v7.5 single-query path,
# wrapped as a function so the master loop in the next cell can call it
# for every query without any per-id branching.
def run_channels(QUERY_TEXT, targets, q_emb_raw, q_emb_enriched, q_emb_hyde, gold_doc_set, total_gold, verbose=False):
    # Build canonical sets from LLM targets
    llm_statute_canons = set()
    for raw in targets.get("statute_targets", []) or []:
        c = statute_anchor_canonical(raw)
        if c: llm_statute_canons.add(c)
    co_expanded_canons = set(llm_statute_canons)
    for canon in llm_statute_canons:
        for nb, _ in co_neighbours.get(canon, []):
            co_expanded_canons.add(nb)

    # v7.5: corpus-derived code-family expansion for per_area_bedrock.
    statute_target_codes = set()
    for canon in llm_statute_canons:
        if " " in canon: statute_target_codes.add(canon.split()[1])
    llm_codes_only = set(statute_target_codes)
    _kfam = CONFIG.get("code_family_top_k", 8)
    for c in llm_codes_only:
        related = sorted(
            ((cc, n) for (a, cc), n in code_pair_count.items() if a == c),
            key=lambda x: -x[1])[:_kfam]
        for cc, _ in related:
            statute_target_codes.add(cc)

    # Concept expansion (weighted exact + substring + token-overlap)
    llm_concepts = (targets.get("concept_targets_en") or []) + (targets.get("legal_area_keywords") or [])
    corpus_concept_keys = set(idx_concept_en.keys())
    expanded_weighted = expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25)

    # Topical channels
    ch_law_direct = channel_law_direct(co_expanded_canons, idx_law_direct, CONFIG["budget_law_direct"])
    ch_court_stat = channel_court_statute(llm_statute_canons, idx_court_statute, idx_court_statute_count, doc_meta, CONFIG["budget_court_statute"])
    ch_concept    = channel_concept(expanded_weighted, idx_concept_en, CONFIG["budget_concept"])
    ch_term       = channel_term(targets, idx_term_orig, idx_term_lemma,
                                  CONFIG["budget_term"], corpus_keys=term_orig_keys)
    ch_per_area   = channel_per_area_bedrock(targets.get("legal_area_keywords", []),
                                              statute_target_codes, per_area_canon_count,
                                              idx_law_direct, CONFIG["budget_per_area"])
    ch_cocit      = channel_co_citation(targets, co_neighbours, idx_law_direct, idx_court_statute,
                                         CONFIG["budget_co_citation"])
    ch_bm25 = bm25_search_multilang(QUERY_TEXT, targets, CONFIG["budget_bm25"])

    ch_vector  = channel_vector(q_emb_raw,      CONFIG["budget_vector"])           if VECTOR_OK else []
    ch_venrich = channel_vector(q_emb_enriched, CONFIG["budget_vector_enriched"])  if VECTOR_OK else []
    ch_vhyde   = channel_vector(q_emb_hyde,     CONFIG["budget_vector_hyde"])      if VECTOR_OK else []

    def _court_hits(hits): return {d for d, _ in hits if doc_meta.get(d, {}).get("family") == "court"}
    seed = (
        _court_hits(ch_court_stat) | _court_hits(ch_law_direct)
      | _court_hits(ch_concept)    | _court_hits(ch_term)
      | _court_hits(ch_per_area)   | _court_hits(ch_cocit)
      | _court_hits(ch_bm25)
      | _court_hits(ch_vector)     | _court_hits(ch_venrich)
      | _court_hits(ch_vhyde)
    )

    ch_sibling = channel_sibling(seed, idx_court_base, idx_judgment_importance, doc_meta, CONFIG["budget_sibling"])
    if GRAPH_OK:
        ch_graph_fwd = channel_graph_forward(seed, idx_graph_out, idx_judgment_importance, doc_meta, CONFIG["budget_graph_forward"])
        ch_graph_rev = channel_graph_reverse(seed, idx_graph_in, idx_judgment_importance, doc_meta, CONFIG["budget_graph_reverse"])
        ch_graph_2h  = channel_graph_2hop(seed, idx_graph_out, CONFIG["budget_graph_2hop"]) if CONFIG["enable_graph_2hop"] else []
    else:
        ch_graph_fwd = []; ch_graph_rev = []; ch_graph_2h = []

    backprop_seed = seed | _court_hits(ch_sibling) | _court_hits(ch_graph_fwd) | _court_hits(ch_graph_rev)
    ch_backprop = channel_statute_backprop(backprop_seed, doc_statute_anchors, idx_law_direct, CONFIG["budget_backprop"])

    CHANNELS = [
        ("law_direct_match",  ch_law_direct),
        ("court_statute",     ch_court_stat),
        ("co_citation",       ch_cocit),
        ("per_area_bedrock",  ch_per_area),
        ("statute_backprop",  ch_backprop),
        ("sibling_expansion", ch_sibling),
        ("graph_forward",     ch_graph_fwd),
        ("graph_reverse",     ch_graph_rev),
        ("graph_2hop",        ch_graph_2h),
        ("concept_en",        ch_concept),
        ("term_orig",         ch_term),
        ("bm25",              ch_bm25),
        ("vector_raw",        ch_vector),
        ("vector_enriched",   ch_venrich),
        ("vector_hyde",       ch_vhyde),
    ]

    channel_recalls = {}
    for name, hits in CHANNELS:
        found = {d for d, _ in hits}
        g = len(found & gold_doc_set) if gold_doc_set else 0
        channel_recalls[name] = (g, len(hits), g / max(1, total_gold))

    union_did = set()
    for _, hits in CHANNELS:
        union_did.update(d for d, _ in hits)
    union_gold = len(union_did & gold_doc_set) if gold_doc_set else 0

    if verbose:
        print(f"  {'channel':<22}  {'size':>6}  {'gold_in_ch':>11}  recall")
        print("  " + "-" * 58)
        for name in [c for c, _ in CHANNELS]:
            g, sz, r = channel_recalls[name]
            print(f"  {name:<22}  {sz:>6}  {g:>11}  {100*r:5.1f}%")
        print(f"  Union: {len(union_did):,} unique doc_ids   gold-in-union: {union_gold}/{total_gold}")

    return {
        "channels": CHANNELS,
        "channel_recalls": channel_recalls,
        "union_size": len(union_did),
        "union_gold": union_gold,
        "seed_size": len(seed),
    }

print("run_channels() defined.")


run_channels() defined.


# Phase 9 — RRF fusion + master loop

Define `fuse_for_query(...)` and then run the full pipeline (channels + fusion + gate) for every query in `ALL_QUERIES`. Results accumulate into `PER_QUERY[qid]`.


In [18]:
def rrf_fuse(channels, k, weights=None):
    """v7.4 weighted RRF: score(did) = sum_ch weights[ch] / (k + rank + 1)."""
    if weights is None: weights = {}
    score = defaultdict(float)
    for name, hits in channels:
        w = weights.get(name, 1.0)
        if w == 0: continue
        for rank, (did, _) in enumerate(hits):
            score[did] += w / (k + rank + 1)
    return score

SUBSTANTIVE_ROLES = {
    "facts", "reasoning", "legal_standard", "application",
    "holding", "citation", "procedural_history",
}

def apply_neg_gate(doc_ids, doc_meta, noise_roles):
    keep = []
    for did in doc_ids:
        m = doc_meta.get(did) or {}
        pr = (m.get("paragraph_role") or "").lower()
        if pr in SUBSTANTIVE_ROLES:
            keep.append(did); continue
        if m.get("is_notification_paragraph"): continue
        if pr in noise_roles: continue
        keep.append(did)
    return keep

def round_robin_guarantee(channels_by_name, guarantee_channel_names,
                           per_channel_cap, total_cap):
    iters = {cn: iter(channels_by_name.get(cn, [])) for cn in guarantee_channel_names}
    counts = {cn: 0 for cn in guarantee_channel_names}
    out = []; seen = set()
    while iters and len(out) < total_cap:
        exhausted = []
        for cn in list(iters.keys()):
            if counts[cn] >= per_channel_cap:
                exhausted.append(cn); continue
            try:
                did, _ = next(iters[cn])
                while did in seen:
                    did, _ = next(iters[cn])
                out.append(did); seen.add(did); counts[cn] += 1
                if len(out) >= total_cap: break
            except StopIteration:
                exhausted.append(cn)
        for cn in exhausted:
            if cn in iters: del iters[cn]
    return out

# R@K sampling points — fixed list, identical for every query.
K_SAMPLES = [50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000,
             5000, 7500, 10000, 15000, 20000, 25000, 35000, 50000]

def fuse_for_query(CHANNELS, gold_doc_set, total_gold):
    rrf_scores = rrf_fuse(CHANNELS, CONFIG["rrf_k"], weights=CONFIG.get("channel_weights"))
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    ranked_dids = [d for d, _ in ranked]
    ranked_gated = apply_neg_gate(ranked_dids, doc_meta, CONFIG["noise_paragraph_roles"])

    ch_by_name = dict(CHANNELS)
    guarantee = round_robin_guarantee(
        ch_by_name, CONFIG["guarantee_channels"],
        CONFIG.get("guarantee_per_channel", 400),
        CONFIG["topk_final"],
    )
    guarantee = apply_neg_gate(guarantee, doc_meta, CONFIG["noise_paragraph_roles"])

    PASS_K = CONFIG["topk_final"]
    final_topk = list(guarantee); seen_f = set(final_topk)
    for did in ranked_gated:
        if len(final_topk) >= PASS_K: break
        if did not in seen_f:
            final_topk.append(did); seen_f.add(did)
    final_topk = final_topk[:PASS_K]

    all_ranked = guarantee + [d for d in ranked_gated if d not in set(guarantee)]
    curve = {}
    for K in K_SAMPLES:
        K_use = min(K, len(all_ranked))
        g = len(set(all_ranked[:K_use]) & gold_doc_set) if gold_doc_set else 0
        curve[K_use] = (g, g / max(1, total_gold))

    union_after_gate = set(all_ranked)
    return {
        "final_topk": final_topk,
        "gold_in_top": len(set(final_topk) & gold_doc_set) if gold_doc_set else 0,
        "R_at_K": (len(set(final_topk) & gold_doc_set) / max(1, total_gold)) if gold_doc_set else 0.0,
        "curve": curve,
        "pre_gate_size": len(ranked_dids),
        "post_gate_size": len(ranked_gated),
        "guarantee_size": len(guarantee),
        "union_after_gate_size": len(union_after_gate),
        "union_after_gate_gold": len(union_after_gate & gold_doc_set) if gold_doc_set else 0,
    }

# Master loop — iterate every query in ALL_QUERIES with identical code path.
import time as _time
print("=" * 80)
print(f"Running pipeline on {len(ALL_QUERIES)} queries  (topk_final={CONFIG['topk_final']:,})")
print("=" * 80)
PER_QUERY = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    g_set = ALL_GOLD_DOC_SET[qid]
    g_tot = ALL_TOTAL_GOLD[qid]
    t0 = _time.time()
    print(f"\n[{qid}]  gold={g_tot}  gold_doc_ids={len(g_set)}")
    ch_out = run_channels(q["query_text"], ALL_TARGETS[qid],
                          ALL_Q_EMB_RAW[qid], ALL_Q_EMB_ENRICHED[qid],
                          ALL_Q_EMB_HYDE[qid],
                          g_set, g_tot, verbose=True)
    fuse_out = fuse_for_query(ch_out["channels"], g_set, g_tot)
    # Per-channel doc_id sets — needed for missed-gold diagnostic.
    chan_hit_sets = {name: {d for d, _ in hits} for name, hits in ch_out["channels"]}
    union_did_set = set()
    for s in chan_hit_sets.values():
        union_did_set |= s
    missed_gold = (g_set - union_did_set) if g_set else set()
    PER_QUERY[qid] = {
        "gold": g_tot,
        "gold_doc_ids": len(g_set),
        "channel_recalls": ch_out["channel_recalls"],
        "channel_hit_sets": chan_hit_sets,
        "union_did_set": union_did_set,
        "missed_gold_dids": sorted(missed_gold),
        "union_size": ch_out["union_size"],
        "union_gold": ch_out["union_gold"],
        "post_gate_union_size": fuse_out["union_after_gate_size"],
        "post_gate_union_gold": fuse_out["union_after_gate_gold"],
        "gold_in_top": fuse_out["gold_in_top"],
        "R_at_K": fuse_out["R_at_K"],
        "curve": fuse_out["curve"],
        "final_topk": fuse_out["final_topk"],
    }
    print(f"  -> R@{CONFIG['topk_final']:,} = {fuse_out['R_at_K']:.3f}  "
          f"({fuse_out['gold_in_top']}/{g_tot})    "
          f"post-gate union gold = {fuse_out['union_after_gate_gold']}/{g_tot}    "
          f"({_time.time()-t0:.1f}s)")


Running pipeline on 10 queries  (topk_final=50,000)

[val_001]  gold=42  gold_doc_ids=42
  channel                   size   gold_in_ch  recall
  ----------------------------------------------------------
  law_direct_match           121            7   16.7%
  court_statute             7852           12   28.6%
  co_citation               2500            0    0.0%
  per_area_bedrock          1500           17   40.5%
  statute_backprop          2000           17   40.5%
  sibling_expansion         5000            4    9.5%
  graph_forward             5000            7   16.7%
  graph_reverse             3000            0    0.0%
  graph_2hop                   0            0    0.0%
  concept_en                3000            7   16.7%
  term_orig                 2500            4    9.5%
  bm25                      2000            5   11.9%
  vector_raw                2000           13   31.0%
  vector_enriched           2000           13   31.0%
  vector_hyde               2000        

# Phase 10 — Aggregate per-query table


In [19]:
# Aggregation header — per-query table + macro mean.
print("=" * 80)
print(f"AGGREGATE SUMMARY  (topk_final={CONFIG['topk_final']:,})")
print("=" * 80)
print()
hdr = f"{'query':<10}  {'gold':>4}  {'R@K':>7}  {'caught':>9}  {'union':>9}  {'gate_union':>11}"
print(hdr)
print("-" * len(hdr))
sum_R = 0.0; sum_union_gold = 0; sum_gate_union_gold = 0; sum_gold = 0
for qid in sorted(PER_QUERY.keys()):
    r = PER_QUERY[qid]
    print(f"{qid:<10}  {r['gold']:>4}  {r['R_at_K']:>7.3f}  "
          f"{r['gold_in_top']:>3}/{r['gold']:<5}  "
          f"{r['union_gold']:>3}/{r['gold']:<5}  "
          f"{r['post_gate_union_gold']:>3}/{r['gold']:<7}")
    sum_R += r["R_at_K"]
    sum_union_gold += r["union_gold"]
    sum_gate_union_gold += r["post_gate_union_gold"]
    sum_gold += r["gold"]
n = max(1, len(PER_QUERY))
print("-" * len(hdr))
print(f"{'MEAN':<10}        {sum_R/n:>7.3f}                  "
      f"{sum_union_gold:>3}/{sum_gold:<5}  "
      f"{sum_gate_union_gold:>3}/{sum_gold:<7}   "
      f"(micro union={sum_union_gold/max(1,sum_gold):.3f}, "
      f"gate union={sum_gate_union_gold/max(1,sum_gold):.3f})")


AGGREGATE SUMMARY  (topk_final=50,000)

query       gold      R@K     caught      union   gate_union
------------------------------------------------------------
val_001       42    0.929   39/42      39/42      39/42     
val_002       36    0.778   28/36      28/36      28/36     
val_003       47    0.745   35/47      35/47      35/47     
val_004       10    1.000   10/10      10/10      10/10     
val_005       11    1.000   11/11      11/11      11/11     
val_006       18    0.944   17/18      17/18      17/18     
val_007       19    0.895   17/19      17/19      17/19     
val_008       29    0.828   24/29      24/29      24/29     
val_009       14    0.857   12/14      12/14      12/14     
val_010       25    0.880   22/25      22/25      22/25     
------------------------------------------------------------
MEAN                0.885                  215/251    215/251       (micro union=0.857, gate union=0.857)


## 10.2 Macro mean R@K curve


In [20]:
# Macro mean R@K curve across all queries.
print("=" * 80)
print("MACRO MEAN R@K CURVE (averaged across all queries)")
print("=" * 80)
all_K = sorted({K for r in PER_QUERY.values() for K in r["curve"].keys()})
print(f"{'K':>6}  {'mean recall':>12}  {'min':>7}  {'max':>7}")
print("-" * 40)
for K in all_K:
    rs = []
    for r in PER_QUERY.values():
        keys = sorted(r["curve"].keys())
        kk = max((k for k in keys if k <= K), default=None)
        if kk is not None:
            rs.append(r["curve"][kk][1])
    if rs:
        print(f"{K:>6}  {sum(rs)/len(rs):>11.3f}   "
              f"{min(rs):>6.3f}  {max(rs):>6.3f}")


MACRO MEAN R@K CURVE (averaged across all queries)
     K   mean recall      min      max
----------------------------------------
    50        0.085    0.000   0.211
   100        0.095    0.000   0.300
   200        0.202    0.056   0.600
   300        0.280    0.128   0.600
   500        0.351    0.149   0.643
   750        0.418    0.170   0.643
  1000        0.495    0.213   0.800
  1500        0.564    0.277   0.900
  2000        0.619    0.362   0.900
  3000        0.659    0.447   1.000
  5000        0.724    0.532   1.000
  7500        0.791    0.574   1.000
 10000        0.815    0.596   1.000
 15000        0.865    0.702   1.000
 20000        0.882    0.745   1.000
 25000        0.885    0.745   1.000
 25118        0.885    0.745   1.000
 25599        0.885    0.745   1.000
 25961        0.885    0.745   1.000
 26954        0.885    0.745   1.000
 27381        0.885    0.745   1.000
 29645        0.885    0.745   1.000
 29982        0.885    0.745   1.000
 30011        0.88

## 10.3 Per-channel mean recall


In [21]:
# Per-channel mean recall across all queries.
print("=" * 80)
print("PER-CHANNEL MEAN RECALL (across all queries)")
print("=" * 80)
all_channels = list(next(iter(PER_QUERY.values()))["channel_recalls"].keys())
print(f"{'channel':<22}  {'mean recall':>11}  {'mean size':>10}")
print("-" * 50)
ranked = []
for cn in all_channels:
    rs = [PER_QUERY[qid]["channel_recalls"].get(cn, (0, 0, 0.0))[2] for qid in PER_QUERY]
    sizes = [PER_QUERY[qid]["channel_recalls"].get(cn, (0, 0, 0.0))[1] for qid in PER_QUERY]
    ranked.append((cn, sum(rs)/len(rs), sum(sizes)/len(sizes)))
ranked.sort(key=lambda x: -x[1])
for cn, r, sz in ranked:
    print(f"{cn:<22}  {r:>10.3f}  {sz:>9.0f}")


PER-CHANNEL MEAN RECALL (across all queries)
channel                 mean recall   mean size
--------------------------------------------------
statute_backprop             0.586       2000
graph_forward                0.307       5000
per_area_bedrock             0.285        982
vector_hyde                  0.273       1998
vector_enriched              0.220       2000
vector_raw                   0.213       2000
law_direct_match             0.141         84
concept_en                   0.097       3000
bm25                         0.097       2000
court_statute                0.079       4130
term_orig                    0.058       2500
co_citation                  0.022       1953
sibling_expansion            0.021       5000
graph_reverse                0.006       3000
graph_2hop                   0.000          0


# Phase 10.4 — Missed-gold diagnostic

For every query where R@K < 1.0, list the gold doc_ids that NO channel returned (i.e. they fell through every one of the 14 channels). For each, print citation, family (law/court), language, court_base, paragraph_role, and a text excerpt — so we can see exactly what signal type would be needed to recover them.

The point is to choose the next fix based on observed misses, not speculation.


In [22]:
# Missed-gold diagnostic — what fell through every channel?
print("=" * 80)
print("MISSED-GOLD DIAGNOSIS  (gold doc_ids absent from every channel)")
print("=" * 80)

# Aggregate breakdown buckets.
from collections import Counter as _C
agg_family = _C(); agg_lang = _C(); agg_role = _C(); agg_courtbase_kind = _C()
total_missed = 0

for qid in sorted(PER_QUERY.keys()):
    r = PER_QUERY[qid]
    missed = r.get("missed_gold_dids", [])
    if not missed:
        print(f"\n[{qid}]  R@K={r['R_at_K']:.3f}   no missed gold")
        continue
    total_missed += len(missed)
    print(f"\n[{qid}]  R@K={r['R_at_K']:.3f}   gold={r['gold']}   "
          f"missed={len(missed)}   (channels all dark for these doc_ids)")
    for did in missed:
        m = doc_meta.get(did, {}) or {}
        cit  = m.get("citation", did)
        fam  = m.get("family", "?")
        lang = (m.get("language") or m.get("language_code") or "?")
        cb   = m.get("court_base") or ""
        pr   = (m.get("paragraph_role") or "")
        txt  = (search_text.get(did, "") or "")[:300].replace("\n", " ")
        agg_family[fam] += 1
        agg_lang[lang] += 1
        agg_role[pr or "(none)"] += 1
        # Court-base kind: e.g. BGE / 1B / 1C / 2C / district court / etc.
        if cb:
            kind = cb.split("_")[0] if "_" in cb else (cb.split()[0] if " " in cb else cb)
            agg_courtbase_kind[kind] += 1
        else:
            agg_courtbase_kind["(no court_base)"] += 1
        print(f"   - {cit}")
        print(f"       family={fam}  lang={lang}  court_base={cb}  paragraph_role={pr!r}")
        if txt:
            print(f"       text: {txt}")

print()
print("=" * 80)
print(f"AGGREGATE MISSED-GOLD BREAKDOWN  (total missed across all queries: {total_missed})")
print("=" * 80)
print("\nBy family:")
for k, n in agg_family.most_common():
    print(f"  {k:<10}  {n}  ({100*n/max(1,total_missed):.1f}%)")
print("\nBy language:")
for k, n in agg_lang.most_common():
    print(f"  {k:<10}  {n}  ({100*n/max(1,total_missed):.1f}%)")
print("\nBy paragraph_role:")
for k, n in agg_role.most_common():
    print(f"  {k:<25}  {n}  ({100*n/max(1,total_missed):.1f}%)")
print("\nBy court_base prefix (BGE / 1B / 5A / ... — empty means law row):")
for k, n in agg_courtbase_kind.most_common():
    print(f"  {k:<20}  {n}  ({100*n/max(1,total_missed):.1f}%)")


MISSED-GOLD DIAGNOSIS  (gold doc_ids absent from every channel)

[val_001]  R@K=0.929   gold=42   missed=3   (channels all dark for these doc_ids)
   - 7B_496/2025 E. 3.2
       family=court  lang=?  court_base=7B_496/2025  paragraph_role='procedural_history'
       text: 7B_496/2025 E. 3.2 3.2. Die Vorinstanz sieht weiterhin Kollusionsmöglichkeiten für den Beschwerdeführer. Zwar sei die Strafuntersuchung bereits weit fortgeschritten. Dennoch seien die zwei Mobiltelefone des Beschwerdeführers noch immer versiegelt und das Entsiegelungsverfahren sei noch im Gang. Bei 
   - Art. 37 Abs. 1 StBOG
       family=law  lang=?  court_base=  paragraph_role=''
       text: Art. 37 Abs. 1 StBOG The federal criminal courts' appellate chambers decide appeals referred to them by the StPO as competent. Federal criminal courts' appellate chambers decide appeals referred to them by the StPO. Which appeals are decided by the federal criminal courts' appellate chambers? Appeal
   - Art. 39 Abs. 1 StBOG
  

# Phase 11 — Save artifacts

Writes one `final_topk_<qid>.json` per query plus an aggregate `summary_multiquery.json`.


In [23]:
import json as _json
out_dir = PATHS["out_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

did_to_cit = {did: m.get("citation", did) for did, m in doc_meta.items()}

# Per-query final top-K
for qid, r in PER_QUERY.items():
    final_top_records = [
        {"rank": i, "doc_id": did, "citation": did_to_cit.get(did, did)}
        for i, did in enumerate(r["final_topk"])
    ]
    (out_dir / f"final_topk_{qid}.json").write_text(
        _json.dumps(final_top_records, ensure_ascii=False), encoding="utf-8")

# Aggregate summary
n = max(1, len(PER_QUERY))
sum_gold = sum(r["gold"] for r in PER_QUERY.values())
sum_caught = sum(r["gold_in_top"] for r in PER_QUERY.values())
sum_union = sum(r["union_gold"] for r in PER_QUERY.values())
sum_gate_union = sum(r["post_gate_union_gold"] for r in PER_QUERY.values())
summary = {
    "topk_final": CONFIG["topk_final"],
    "n_queries": len(PER_QUERY),
    "macro_mean_R_at_K": sum(r["R_at_K"] for r in PER_QUERY.values()) / n,
    "micro_R_at_K": sum_caught / max(1, sum_gold),
    "micro_union_recall": sum_union / max(1, sum_gold),
    "micro_gate_union_recall": sum_gate_union / max(1, sum_gold),
    "per_query": {
        qid: {
            "gold": r["gold"],
            "gold_doc_ids": r["gold_doc_ids"],
            "R_at_K": r["R_at_K"],
            "gold_in_top": r["gold_in_top"],
            "union_gold": r["union_gold"],
            "post_gate_union_gold": r["post_gate_union_gold"],
            "channel_recalls": {
                cn: {"gold": v[0], "size": v[1], "recall": v[2]}
                for cn, v in r["channel_recalls"].items()
            },
            "curve": {str(K): {"gold": v[0], "recall": v[1]} for K, v in r["curve"].items()},
        }
        for qid, r in PER_QUERY.items()
    },
}
(out_dir / "summary_multiquery.json").write_text(
    _json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

(out_dir / "targets_multiquery.json").write_text(
    _json.dumps(ALL_TARGETS, ensure_ascii=False, indent=2), encoding="utf-8")

(out_dir / "config.json").write_text(_json.dumps(
    {k: v for k, v in CONFIG.items() if not isinstance(v, set)},
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print(f"Wrote artifacts to {out_dir}")
print(f"  summary_multiquery.json")
print(f"  targets_multiquery.json")
print(f"  config.json")
print(f"  final_topk_<query_id>.json   x {len(PER_QUERY)}")


Wrote artifacts to /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7
  summary_multiquery.json
  targets_multiquery.json
  config.json
  final_topk_<query_id>.json   x 10


## 11.2 Cleanup (free GPU memory)

**What:** Delete large GPU tensors so a notebook re-run starts clean.

**Why:** Colab keeps state across cells; without explicit cleanup, re-running
Phase 6 will OOM.

In [24]:
import gc
try:
    del E_GPU
except NameError:
    pass
try:
    del EMB_MODEL
except NameError:
    pass
gc.collect()
import torch as _torch2
if _torch2.cuda.is_available():
    _torch2.cuda.empty_cache()
    print(f"VRAM after cleanup: {_torch2.cuda.memory_allocated()/1024**3:.2f} GB")
print("cleaned up.")

VRAM after cleanup: 0.01 GB
cleaned up.
